# RSNA Knee — rsna-knee-ens-cnx-pub80-r384

**Sans TTA.** Internet off, **deux T4**. Pas de flip.

1. Mix lineaire de 5 modeles, poids par finding
   (ViT-k320 / v20b / v20h distil / CoAt-a / CNX-r7).
   Les 4 premiers gardent les proportions du mix publie.
   CNX-r7 prend, par finding, la part du softmax hybrid T=0.05
   (`subset_cnx_r7_in22k_v3b_hybrid025`, epoch 7). Le reste est
   renormalise sur les 4.
   ViT-k320 : `vit_k_llrd08_320_v3b_oofmask_v20h_full_oai3` SWA 1-3
   (gold 0.990). Entree 320, FOV 130 mm.
   v20b DINO : SWA OAI 1-3 (gold 0.970). Entree 224, FOV 130 mm.
   v20h distil : `v20h_crop145_dino_ep11_v3b_rsna_distil_main_full` SWA 7-9
   (gold 0.969). Entree 224, FOV 145 mm.
   CoAt-a : SWA OAI 1-3 (gold 0.988). Entree 224, FOV 130 mm.
   CNX-r7 : `cnx_r7_in22k_v3b_oofmask_full` SWA 7-10 (gold 0.976).
   Entree 224, FOV 130 mm. ConvNeXt-Small IN-22k.
   Les etudes sont coupees sur les deux T4. Une seule etude est decodee
   a la fois. **Budget 6,75 h** (1 300 etudes a ~17 s en demandent 6,1 h),
   puis la memoire est rendue.
2. Raptor-384 seul (`raptor_ft_alldata_t16_blendjev_oai_d96_r384_swa`).
   Pas de Raptor-224, pas du reste du notebook public. Timeout 2 h.
3. Fusion par rangs, **80 %** le mix des 5 modeles et **20 %** Raptor-384.


In [ ]:
import time
NB_T0 = time.time()
MIX5_BUDGET_S = 6.75 * 3600
print('horloge depart', flush=True)
print('budget mix5 h', MIX5_BUDGET_S / 3600, '| raptor384 timeout h 2', flush=True)


In [ ]:
import os
from pathlib import Path

KAGGLE = os.path.exists('/kaggle')
if KAGGLE:
    DATA_DIR = Path('/kaggle/input')
    print('contenu /kaggle/input :', sorted(p.name for p in DATA_DIR.iterdir()))
    COMP_DIR = None
    for base in [DATA_DIR / 'competitions', DATA_DIR]:
        if base.exists():
            for p in sorted(base.iterdir()):
                if (p / 'test.csv').exists():
                    COMP_DIR = p
                    break
        if COMP_DIR is not None:
            break
    if COMP_DIR is None:
        raise FileNotFoundError('test.csv introuvable sous /kaggle/input')
    search_roots = []
    if (DATA_DIR / 'datasets').exists():
        search_roots.append(DATA_DIR / 'datasets')
    search_roots.append(DATA_DIR)

    def _find_weight(name):
        for root in search_roots:
            hits = list(root.rglob(name))
            if hits:
                return hits[0]
        raise FileNotFoundError(name)

    WEIGHTS_VIT = _find_weight('vit_k320_oai3_swa_ep1_3.pt')
    WEIGHTS_V20H = _find_weight('v20h_crop145_distil_swa_ep7_9.pt')
    WEIGHTS_V20B = _find_weight('v20b_dino_oai3_swa_ep1_3.pt')
    WEIGHTS_COAT = _find_weight('coat_a_oai3_swa_ep1_3.pt')
    WEIGHTS_CNX = _find_weight('cnx_r7_swa_ep7_10.pt')
else:
    COMP_DIR = Path('../data/csv')
    WEIGHTS_VIT = Path('kaggle_dataset_vit_k/_upload_k320_oai3_swa/vit_k320_oai3_swa_ep1_3.pt')
    WEIGHTS_V20H = Path('kaggle_dataset_v20h_lat/_upload_crop145_distil_swa789/v20h_crop145_distil_swa_ep7_9.pt')
    WEIGHTS_V20B = Path('kaggle_dataset_v20b/_upload_dino_oai3_swa/v20b_dino_oai3_swa_ep1_3.pt')
    WEIGHTS_COAT = Path('kaggle_dataset_coat/_upload_oai3_swa/coat_a_oai3_swa_ep1_3.pt')
    WEIGHTS_CNX = Path('kaggle_dataset_cnx/_upload_swa78910/cnx_r7_swa_ep7_10.pt')

TEST_CSV = COMP_DIR / 'test.csv'
TEST_SERIES_CSV = COMP_DIR / 'test_series.csv'
TEST_SERIES_DIR = COMP_DIR / 'test_series'
SAMPLE_SUB = COMP_DIR / 'sample_submission.csv'

import torch

def _pick_device():
    if not torch.cuda.is_available():
        return 'cpu'
    try:
        m = torch.nn.Conv2d(3, 8, 3, padding=1).cuda()
        _ = m(torch.zeros(1, 3, 32, 32, device='cuda')).sum().item()
        return 'cuda'
    except Exception as e:
        print('CUDA inutilisable, repli CPU :', str(e)[:120])
        return 'cpu'

DEVICE = _pick_device()
if DEVICE == 'cuda':
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
try:
    torch.set_float32_matmul_precision('high')
except Exception:
    pass

AMP_DTYPE = (torch.bfloat16
             if (DEVICE == 'cuda' and torch.cuda.is_bf16_supported())
             else torch.float16)

# Ordre : ViT-k320, v20b, v20h distil, CoAt-a, CNX-r7. Somme 1, plancher 0.
N_MODELS = 5
MIX = [
    (0.07, 0.27, 0.46, 0.03, 0.17),  # ACL
    (0.03, 0.43, 0.51, 0.01, 0.02),  # MCL
    (0.20, 0.22, 0.23, 0.11, 0.24),  # Medial Meniscus
    (0.14, 0.29, 0.33, 0.04, 0.20),  # Lateral Meniscus
    (0.13, 0.25, 0.26, 0.14, 0.22),  # Medial OA
    (0.16, 0.22, 0.22, 0.16, 0.24),  # Lateral OA
    (0.12, 0.27, 0.37, 0.10, 0.14),  # PF OA
    (0.22, 0.18, 0.17, 0.21, 0.22),  # Effusion
    (0.26, 0.22, 0.21, 0.17, 0.14),  # Synovitis
    (0.21, 0.22, 0.26, 0.13, 0.18),  # Baker's
    (0.15, 0.24, 0.28, 0.12, 0.21),  # Contusion
    (0.19, 0.16, 0.22, 0.21, 0.22),  # Fracture
]

print('competition dir :', COMP_DIR)
print('poids vit       :', WEIGHTS_VIT, '| existe :', WEIGHTS_VIT.exists())
print('poids v20h dino :', WEIGHTS_V20H, '| existe :', WEIGHTS_V20H.exists())
print('poids v20b dino :', WEIGHTS_V20B, '| existe :', WEIGHTS_V20B.exists())
print('poids coat      :', WEIGHTS_COAT, '| existe :', WEIGHTS_COAT.exists())
print('poids cnx       :', WEIGHTS_CNX, '| existe :', WEIGHTS_CNX.exists())
print('device          :', DEVICE)
print('TTA             : non')


In [ ]:
# --- Pretraitement DICOM -> volume (24, 224, 224) uint8 --------------------
# (copie self-contained de src/infer/preprocess_dicom.py)
# crop 130 mm + bande 20-80 % (air / bords), identique a la conversion NPZ
import cv2
import numpy as np
import pydicom

N_SLICES, IMG_SIZE = 24, 224
CROP_MM, CROP_REL, BAND = 130.0, 0.70, (0.20, 0.80)


def _slice_sort_key(ds):
    inst = getattr(ds, 'InstanceNumber', None)
    if inst is not None:
        try:
            return float(inst)
        except (TypeError, ValueError):
            pass
    ipp = getattr(ds, 'ImagePositionPatient', None)
    iop = getattr(ds, 'ImageOrientationPatient', None)
    if ipp is not None and iop is not None:
        row = np.array(iop[:3], dtype=np.float64)
        col = np.array(iop[3:], dtype=np.float64)
        normal = np.cross(row, col)
        return float(np.dot(np.array(ipp, dtype=np.float64), normal))
    return float(ipp[2]) if ipp is not None else 0.0


def _spacing_mm(ds):
    ps = getattr(ds, 'PixelSpacing', None) or getattr(ds, 'ImagerPixelSpacing', None)
    if ps is None:
        return None
    try:
        return float(ps[0]), float(ps[1])
    except (TypeError, ValueError, IndexError):
        return None


def _center_crop(img, row_mm, col_mm):
    h, w = img.shape[:2]
    if row_mm is not None and col_mm is not None and row_mm > 0 and col_mm > 0:
        ch = min(h, max(8, int(round(CROP_MM / row_mm))))
        cw = min(w, max(8, int(round(CROP_MM / col_mm))))
    else:
        ch = min(h, max(8, int(round(h * CROP_REL))))
        cw = min(w, max(8, int(round(w * CROP_REL))))
    y0, x0 = (h - ch) // 2, (w - cw) // 2
    return img[y0:y0 + ch, x0:x0 + cw]


_SERIES_CACHE = {}

def load_series_dicom(series_dir, img_size=224):
    key = str(Path(series_dir))
    if key not in _SERIES_CACHE:
        _SERIES_CACHE[key] = _series_band(series_dir)
    band = _SERIES_CACHE[key]
    size = int(img_size)
    out = np.empty((band.shape[0], size, size), dtype=np.uint8)
    for i in range(band.shape[0]):
        sl = band[i]
        if sl.shape != (size, size):
            sl = cv2.resize(sl, (size, size), interpolation=cv2.INTER_AREA)
        out[i] = np.clip(np.asarray(sl, dtype=np.float32) * 255.0, 0, 255).round().astype(np.uint8)
    return out


def _series_band(series_dir):
    files = sorted(Path(series_dir).glob('*.dcm'))
    datasets = []
    for f in files:
        try:
            datasets.append(pydicom.dcmread(str(f)))
        except Exception:
            continue
    if not datasets:
        raise RuntimeError(f'aucun DICOM lisible dans {series_dir}')
    datasets.sort(key=_slice_sort_key)

    frames, sp = [], None
    for ds in datasets:
        arr = ds.pixel_array.astype(np.float32)
        arr = arr * float(getattr(ds, 'RescaleSlope', 1.0) or 1.0) \
                  + float(getattr(ds, 'RescaleIntercept', 0.0) or 0.0)
        if getattr(ds, 'PhotometricInterpretation', 'MONOCHROME2') == 'MONOCHROME1':
            arr = arr.max() - arr
        if sp is None:
            sp = _spacing_mm(ds)
        if arr.ndim == 2:
            frames.append(arr)
        elif arr.ndim == 3:
            frames.extend([arr[i] for i in range(arr.shape[0])])
        else:
            raise RuntimeError(f'ndim inattendu {arr.ndim}')
    if not frames:
        raise RuntimeError('aucune coupe')

    row_mm, col_mm = (sp if sp is not None else (None, None))
    cropped = [_center_crop(im, row_mm, col_mm) for im in frames]
    h0, w0 = cropped[0].shape[:2]
    stacked = []
    for c in cropped:
        if c.shape[:2] != (h0, w0):
            c = cv2.resize(c, (w0, h0), interpolation=cv2.INTER_AREA)
        stacked.append(c)
    vol = np.stack(stacked, axis=0)
    lo, hi = np.percentile(vol, 1.0), np.percentile(vol, 99.0)
    if hi <= lo:
        hi = lo + 1.0
    vol = np.clip((vol - lo) / (hi - lo), 0, 1)

    n = vol.shape[0]
    a = int(np.floor(BAND[0] * (n - 1)))
    b = int(np.ceil(BAND[1] * (n - 1)))
    if b - a + 1 < 8:
        a, b = 0, n - 1
    band = vol[a:b + 1]
    idx = np.linspace(0, band.shape[0] - 1, N_SLICES).round().astype(int)
    band = band[idx]
    return band


PROTOCOL_SLOTS = [
    ('Sagittal', 1),
    ('Coronal', 1),
    ('Axial', 1),
    ('Sagittal', 0),
    ('Coronal', 0),
    ('Axial', 0),
]


def _pick_protocol_row(series_rows, plane, fluid):
    '''Un contraste exact (FS ou T1) sur un plan. Pas de repli.'''
    rows = series_rows[series_rows.Anatomical_Plane == plane]
    if len(rows) == 0:
        return None
    if 'Fluid_Sensitive' in rows.columns:
        rows = rows[rows.Fluid_Sensitive == int(fluid)]
    elif fluid == 1 and 'Fat_Suppression' in rows.columns:
        rows = rows[rows.Fat_Suppression == 1]
    elif fluid == 0 and 'Fat_Suppression' in rows.columns:
        rows = rows[rows.Fat_Suppression == 0]
    if len(rows) == 0:
        return None
    return rows.sort_values('SeriesInstanceUID').iloc[0]


LAT_MIN_OFFSET_MM = 20.0


def _lat_side_tag(v):
    if v is None:
        return None
    s = str(v).strip().upper()
    return s[0] if s and s[0] in ('L', 'R') else None


def _lat_center_x(ipp, iop, ps, rows, cols):
    try:
        ipp = np.asarray(list(ipp), dtype=np.float64)
        iop = np.asarray(list(iop), dtype=np.float64)
        ps = np.asarray(list(ps), dtype=np.float64)
        rows, cols = float(rows), float(cols)
    except (TypeError, ValueError):
        return None
    if ipp.size < 3 or iop.size < 6 or ps.size < 2:
        return None
    c = ipp[:3] + iop[:3] * ps[1] * cols / 2.0 + iop[3:6] * ps[0] * rows / 2.0
    x = float(c[0])
    return x if np.isfinite(x) else None


def laterality_from_study_dir(study_dir):
    '''Canon L : tag Laterality, sinon X centre (zone morte 20 mm).'''
    study_dir = Path(study_dir)
    xs, tags = [], []
    if not study_dir.is_dir():
        return None
    for series in sorted(p for p in study_dir.iterdir() if p.is_dir()):
        files = sorted(series.glob('*.dcm'))
        if not files:
            continue
        try:
            ds = pydicom.dcmread(str(files[len(files) // 2]),
                                 stop_before_pixels=True, force=True)
        except Exception:
            continue
        t = _lat_side_tag(getattr(ds, 'Laterality', None))
        if t:
            tags.append(t)
        x = _lat_center_x(
            getattr(ds, 'ImagePositionPatient', None),
            getattr(ds, 'ImageOrientationPatient', None),
            getattr(ds, 'PixelSpacing', None),
            getattr(ds, 'Rows', None), getattr(ds, 'Columns', None))
        if x is not None:
            xs.append(x)
    if tags:
        return tags[0]
    if not xs:
        return None
    m = float(np.median(xs))
    if abs(m) < LAT_MIN_OFFSET_MM:
        return None
    return 'R' if m < 0 else 'L'


def normalise_laterality(vol, plane, lat):
    if lat != 'R':
        return vol
    plane_c = str(plane or '').strip().capitalize()
    if plane_c in ('Coronal', 'Axial'):
        return np.ascontiguousarray(np.flip(vol, axis=-1))
    if plane_c == 'Sagittal':
        return np.ascontiguousarray(np.flip(vol, axis=0))
    return vol


def study_to_tensor(study_dir, series_rows, n_series=3, laterality=False, img_size=224):
    '''(n_series, 24, img_size, img_size) ; n_series=6 : 3 FS + 3 T1, sans repli.'''
    planes = ['Sagittal', 'Coronal', 'Axial']
    size = int(img_size)
    out = np.zeros((n_series, N_SLICES, size, size), dtype=np.float32)
    used = set()
    lat = laterality_from_study_dir(study_dir) if laterality else None

    def _fill(i, pick, plane):
        if pick is None:
            return
        sdir = Path(study_dir) / pick.SeriesInstanceUID
        if not sdir.exists():
            return
        try:
            vol = load_series_dicom(sdir, img_size=size).astype(np.float32) / 255.0
            if laterality:
                vol = normalise_laterality(vol, plane, lat)
            out[i] = vol
            used.add(str(pick.SeriesInstanceUID))
        except Exception as e:
            print(f'  [warn] {pick.SeriesInstanceUID}: {e}')

    if n_series == 6:
        for i, (plane, fluid) in enumerate(PROTOCOL_SLOTS):
            _fill(i, _pick_protocol_row(series_rows, plane, fluid), plane)
        return out

    for i, plane in enumerate(planes):
        rows = series_rows[series_rows.Anatomical_Plane == plane]
        if len(rows) == 0:
            continue
        fluid = rows[rows.Fluid_Sensitive == 1] if 'Fluid_Sensitive' in rows else rows.iloc[0:0]
        if len(fluid) == 0 and 'Fat_Suppression' in rows:
            fluid = rows[rows.Fat_Suppression == 1]
        pick = (fluid if len(fluid) else rows).sort_values('SeriesInstanceUID').iloc[0]
        _fill(i, pick, plane)
    if n_series >= 4:
        t1 = None
        for plane in planes:
            rows = series_rows[series_rows.Anatomical_Plane == plane]
            if 'Fluid_Sensitive' in rows:
                rows = rows[rows.Fluid_Sensitive == 0]
            rows = rows[~rows.SeriesInstanceUID.isin(used)]
            if len(rows):
                t1 = rows.sort_values('SeriesInstanceUID').iloc[0]
                break
        pl = None if t1 is None else str(t1.Anatomical_Plane)
        _fill(3, t1, pl)
    return out


In [ ]:
# --- KneeNet v20b DINO ---
# -*- coding: utf-8 -*-
"""KneeNet v20b — v17 + 6 slots protocole (FS / T1 par plan).

Base v17 inchangee : ResNet-50 RadImageNet, stem A+B, CAM 14x14 dilated,
BiLSTM residuel init 0, BN gelee. Pas de MHA (c'est v20a). Pas de gate T1.

Entree : (B, 6, 24, H, W) grayscale [0, 1]. Ordre des slots :

  0 SAG_FS   1 COR_FS   2 AX_FS
  3 SAG_T1   4 COR_T1   5 AX_T1

Chaque slot est un contraste fixe : fluide (FS) ou T1, sans repli.
Slot absent = volume zero, masque `valid` (comme un plan droppe en v17).
Le T1 n'a plus a battre le fluide dans un 4e slot unique.

Dans train_series.csv, Fluid_Sensitive == Fat_Suppression : pas de fluide
sans FS. AX_T1 remplace le slot vide SAG_FLUID_NOFS du notebook candidat.

LSTM partage sur les 6 sequences (batch = B*S). Epoch 0 recopie v16.
"""
from __future__ import annotations

from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.checkpoint import checkpoint
from torchvision.models import resnet50

N_TARGETS = 12
DEFAULT_WEIGHTS = "weights/ResNet50.pt"
MAP_HW = 14
LSTM_HIDDEN = 256
N_PROTOCOL_SLOTS = 6


def _dilate_layer4(layer4: nn.Sequential, dilate: int = 2) -> None:
    """stride 32 -> 16, in-place. Poids 3x3 / 1x1 inchanges."""
    for i, block in enumerate(layer4):
        if i == 0:
            block.conv2.stride = (1, 1)
            if block.downsample is not None:
                block.downsample[0].stride = (1, 1)
        block.conv2.dilation = (dilate, dilate)
        block.conv2.padding = (dilate, dilate)


class _ResNet50Encoder(nn.Module):
    """ResNet-50 jusqu'a layer4 dilated. Cles `backbone.N.*` = v14/v16/v17."""

    def __init__(self) -> None:
        super().__init__()
        try:
            base = resnet50(weights=None)
        except TypeError:
            base = resnet50(pretrained=False)
        self.backbone = nn.Sequential(*list(base.children())[:-2])
        _dilate_layer4(self.backbone[7])
        self.num_features = 2048
        for m in self.modules():
            if isinstance(m, nn.ReLU):
                m.inplace = False

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.backbone(x)


class KneeNetV20b(nn.Module):
    arch_id = "kneenet_v20b"

    def __init__(self, backbone: str = "resnet50", n_series: int = N_PROTOCOL_SLOTS,
                 n_targets: int = N_TARGETS, head_dim: int = 512,
                 dropout: float = 0.2, pretrained: bool = True,
                 encode_chunk: int = 16, max_depth: int = 64,
                 backbone_weights: str = DEFAULT_WEIGHTS,
                 lstm_hidden: int = LSTM_HIDDEN) -> None:
        super().__init__()
        self.n_series = n_series
        self.n_targets = n_targets
        self.encode_chunk = int(encode_chunk)
        self.backbone_name = str(backbone)
        self.map_hw = MAP_HW
        self.lstm_hidden = int(lstm_hidden)

        self.encoder = _ResNet50Encoder()
        if pretrained:
            self._load_radimagenet(backbone_weights)
        feat_dim = self.encoder.num_features

        self.plane_embed = nn.Parameter(torch.zeros(n_series, feat_dim))
        self.slice_embed = nn.Parameter(torch.zeros(max_depth, feat_dim))
        self.attn_score = nn.Linear(feat_dim, n_targets)
        self.spatial_score = nn.Conv2d(feat_dim, n_targets, kernel_size=1)
        nn.init.zeros_(self.spatial_score.weight)
        nn.init.zeros_(self.spatial_score.bias)

        self.slice_lstm = nn.LSTM(
            input_size=feat_dim,
            hidden_size=self.lstm_hidden,
            num_layers=1,
            batch_first=True,
            bidirectional=True,
        )
        self.lstm_proj = nn.Linear(2 * self.lstm_hidden, feat_dim)
        nn.init.zeros_(self.lstm_proj.weight)
        nn.init.zeros_(self.lstm_proj.bias)

        self.norm = nn.LayerNorm(feat_dim)
        self.head = nn.Sequential(
            nn.Linear(feat_dim, head_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(head_dim, 1),
        )
        self.stem_mix = nn.Conv2d(3, 3, kernel_size=1, bias=False)
        self._init_stem_mix()
        self.register_buffer(
            "img_mean", torch.zeros(1, 3, 1, 1), persistent=True,
        )
        self.register_buffer(
            "img_std", torch.ones(1, 3, 1, 1), persistent=True,
        )

    def _init_stem_mix(self) -> None:
        """out_c = in_G pour les 3 canaux -> R=G=B=k au depart."""
        with torch.no_grad():
            self.stem_mix.weight.zero_()
            self.stem_mix.weight[:, 1, 0, 0] = 1.0

    def _load_radimagenet(self, path: str) -> None:
        p = Path(path)
        if not p.is_file():
            raise FileNotFoundError(
                f"poids RadImageNet introuvables : {p.resolve()}"
            )
        sd = torch.load(p, map_location="cpu", weights_only=False)
        self.encoder.load_state_dict(sd, strict=True)
        print(
            f"RadImageNet ResNet-50 charge : {p} | CAM {MAP_HW}x{MAP_HW} "
            f"| BiLSTM hidden {self.lstm_hidden} | slots {self.n_series}",
            flush=True,
        )

    def train(self, mode: bool = True):
        super().train(mode)
        for m in self.encoder.modules():
            if isinstance(m, nn.BatchNorm2d):
                m.eval()
        return self

    def encoder_parameters(self):
        return self.encoder.parameters()

    def head_parameters(self):
        for name, p in self.named_parameters():
            if not name.startswith("encoder."):
                yield p

    def _encode_chunks(self, rgb: torch.Tensor) -> torch.Tensor:
        """rgb : (N, 3, H, W) dans [0, 1] -> (N, C, 14, 14)."""
        n = rgb.size(0)
        c = self.encoder.num_features
        if n == 0:
            return rgb.new_zeros(0, c, MAP_HW, MAP_HW)
        outs = []
        chunk = max(1, self.encode_chunk)
        for i in range(0, n, chunk):
            part = rgb[i:i + chunk]
            if self.training:
                feats = checkpoint(self.encoder, part, use_reentrant=False)
            else:
                feats = self.encoder(part)
            outs.append(feats)
        return torch.cat(outs, dim=0)

    def _spatial_pool(self, maps: torch.Tensor, return_attn: bool = False):
        """maps (N, C, H, W) -> gap (N, C), local (N, T, C) [, attn (N, T, H, W)]."""
        n, c, h, w = maps.shape
        gap = maps.mean(dim=(-2, -1))
        logits = self.spatial_score(maps)
        attn = torch.softmax(
            logits.float().reshape(n, self.n_targets, h * w), dim=-1,
        ).to(maps.dtype)
        flat = maps.reshape(n, c, h * w)
        local = torch.einsum("nth,nch->ntc", attn, flat)
        if return_attn:
            return gap, local, attn.reshape(n, self.n_targets, h, w)
        return gap, local

    def _stem_rgb(self, x: torch.Tensor, idx: torch.Tensor) -> torch.Tensor:
        """Triplet A+B pour les tokens valides. x (B,S,D,H,W), idx plat."""
        _b, s, d, _h, _w = x.shape
        bs = idx // (s * d)
        rest = idx % (s * d)
        ss = rest // d
        dd = rest % d
        d_m = (dd - 1).clamp(0, d - 1)
        d_p = (dd + 1).clamp(0, d - 1)
        center = x[bs, ss, dd]
        rgb = torch.stack(
            [
                0.5 + 0.5 * (center - x[bs, ss, d_m]),
                center,
                0.5 + 0.5 * (x[bs, ss, d_p] - center),
            ],
            dim=1,
        )
        rgb = self.stem_mix(rgb)
        return (rgb - self.img_mean) / self.img_std

    def _temporal_delta(self, gaps: torch.Tensor, b: int, s: int,
                        d: int) -> torch.Tensor:
        """gaps (BSD, C) -> residu LSTM (B, S, D, C). Epoch 0 : 0."""
        c = gaps.size(-1)
        seq = gaps.reshape(b * s, d, c)
        with torch.autocast("cuda", enabled=False):
            h, _ = self.slice_lstm(seq.float())
            delta = self.lstm_proj(h)
        return delta.to(dtype=gaps.dtype).reshape(b, s, d, c)

    def _slice_pool(self, x: torch.Tensor, valid: torch.Tensor,
                    gaps: torch.Tensor, local: torch.Tensor):
        """gaps (BSD,C), local (BSD,T,C) -> logits (B,T), slice_attn (B,S,D,T)."""
        b, s, d = valid.shape
        delta = self._temporal_delta(gaps, b, s, d)
        pe = self.plane_embed[:s].view(1, s, 1, -1)
        se = self.slice_embed[:d].view(1, 1, d, -1)
        feats = gaps.reshape(b, s, d, -1) + delta + pe + se
        scores = self.attn_score(feats)
        scores = scores.masked_fill(~valid.unsqueeze(-1), -1.0e4)
        scores = scores.reshape(b, s * d, self.n_targets)
        attn = torch.softmax(scores.float(), dim=1).to(feats.dtype)
        tokens = local.reshape(b, s, d, self.n_targets, -1)
        tokens = tokens + delta.unsqueeze(3) + pe.unsqueeze(3) + se.unsqueeze(3)
        tokens = tokens.reshape(b, s * d, self.n_targets, -1)
        pooled = torch.einsum("bnt,bntc->btc", attn, tokens)
        logits = self.head(self.norm(pooled)).squeeze(-1)
        return logits, attn.reshape(b, s, d, self.n_targets)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x : (B, S, D, H, W) dans [0, 1] -> logits (B, 12)."""
        b, s, d, _h, _w = x.shape
        assert s == self.n_series, f"attendu {self.n_series} series, recu {s}"
        if d > self.slice_embed.size(0):
            raise ValueError(
                f"D={d} > max_depth={self.slice_embed.size(0)} : "
                "augmenter max_depth"
            )

        valid = x.amax(dim=(-2, -1)) > 1e-6
        n_tok = b * s * d
        idx = valid.reshape(-1).nonzero(as_tuple=False).view(-1)
        feat_dim = self.encoder.num_features
        gaps = x.new_zeros(n_tok, feat_dim)
        local = x.new_zeros(n_tok, self.n_targets, feat_dim)
        if idx.numel() > 0:
            maps = self._encode_chunks(self._stem_rgb(x, idx))
            if gaps.dtype != maps.dtype:
                gaps = gaps.to(dtype=maps.dtype)
                local = local.to(dtype=maps.dtype)
            gap, loc = self._spatial_pool(maps)
            gaps[idx] = gap
            local[idx] = loc
        logits, _attn = self._slice_pool(x, valid, gaps, local)
        return logits

    @torch.no_grad()
    def interpret(self, x: torch.Tensor) -> dict[str, torch.Tensor]:
        """Forward eval + cartes. spatial_attn : (B, S, D, T, 14, 14)."""
        self.eval()
        b, s, d, _h, _w = x.shape
        assert s == self.n_series, f"attendu {self.n_series} series, recu {s}"
        valid = x.amax(dim=(-2, -1)) > 1e-6
        n_tok = b * s * d
        idx = valid.reshape(-1).nonzero(as_tuple=False).view(-1)
        feat_dim = self.encoder.num_features
        hw = MAP_HW
        gaps = x.new_zeros(n_tok, feat_dim)
        local = x.new_zeros(n_tok, self.n_targets, feat_dim)
        spatial = x.new_zeros(n_tok, self.n_targets, hw, hw)
        if idx.numel() > 0:
            maps = self._encode_chunks(self._stem_rgb(x, idx))
            if gaps.dtype != maps.dtype:
                gaps = gaps.to(dtype=maps.dtype)
                local = local.to(dtype=maps.dtype)
                spatial = spatial.to(dtype=maps.dtype)
            gap, loc, spat = self._spatial_pool(maps, return_attn=True)
            gaps[idx] = gap
            local[idx] = loc
            spatial[idx] = spat
        logits, slice_attn = self._slice_pool(x, valid, gaps, local)
        delta = self._temporal_delta(gaps, b, s, d)
        pe = self.plane_embed[:s].view(1, s, 1, 1, -1)
        se = self.slice_embed[:d].view(1, 1, d, 1, -1)
        cam_tokens = (
            local.reshape(b, s, d, self.n_targets, -1)
            + delta.unsqueeze(3) + pe + se
        )
        return {
            "logits": logits,
            "slice_attn": slice_attn,
            "spatial_attn": spatial.reshape(
                b, s, d, self.n_targets, hw, hw,
            ),
            "cam_tokens": cam_tokens,
            "valid": valid,
        }


In [ ]:
# --- KneeNet v20h DINO ---
# -*- coding: utf-8 -*-
"""KneeNet v20h — v20b + tete de decision par cible (v20d, 6 slots).

Base v20b inchangee : ResNet-50 RadImageNet, stem A+B, CAM 14x14 dilated,
BiLSTM residuel init 0, BN gelee, 6 slots protocole (FS / T1 par plan).

Levier : v20b partage Linear(C->512) et Linear(512->1) sur les 12 cibles.
Ici, comme v20d (qui n'existait qu'en 3 plans) :

1. `label_embed` (T, C) init 0, ajoute a pooled avant LayerNorm
2. Linear(512 -> T) en **diagonale** : logit_t = W_t · h_t + b_t

A l'init, les T lignes de W (et les T biais) sont copies de la 1re :
epoch 0 = v20b. Les cibles peuvent ensuite diverger.

Entree : (B, 6, 24, H, W) grayscale [0, 1]. Ordre des slots = v20b.
Juger vs subset_v20b_crop130_frac025 (val ET gold, Δ > 0.01), surtout
MCL / Effusion / Medial Meniscus.
"""
from __future__ import annotations

from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.checkpoint import checkpoint
from torchvision.models import resnet50

N_TARGETS = 12
DEFAULT_WEIGHTS = "weights/ResNet50.pt"
MAP_HW = 14
LSTM_HIDDEN = 256
N_PROTOCOL_SLOTS = 6


def _dilate_layer4(layer4: nn.Sequential, dilate: int = 2) -> None:
    """stride 32 -> 16, in-place. Poids 3x3 / 1x1 inchanges."""
    for i, block in enumerate(layer4):
        if i == 0:
            block.conv2.stride = (1, 1)
            if block.downsample is not None:
                block.downsample[0].stride = (1, 1)
        block.conv2.dilation = (dilate, dilate)
        block.conv2.padding = (dilate, dilate)


class _ResNet50Encoder(nn.Module):
    """ResNet-50 jusqu'a layer4 dilated. Cles `backbone.N.*` = v14/v16/v17."""

    def __init__(self) -> None:
        super().__init__()
        try:
            base = resnet50(weights=None)
        except TypeError:
            base = resnet50(pretrained=False)
        self.backbone = nn.Sequential(*list(base.children())[:-2])
        _dilate_layer4(self.backbone[7])
        self.num_features = 2048
        for m in self.modules():
            if isinstance(m, nn.ReLU):
                m.inplace = False

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.backbone(x)


class KneeNetV20h(nn.Module):
    arch_id = "kneenet_v20h"

    def __init__(self, backbone: str = "resnet50", n_series: int = N_PROTOCOL_SLOTS,
                 n_targets: int = N_TARGETS, head_dim: int = 512,
                 dropout: float = 0.2, pretrained: bool = True,
                 encode_chunk: int = 16, max_depth: int = 64,
                 backbone_weights: str = DEFAULT_WEIGHTS,
                 lstm_hidden: int = LSTM_HIDDEN) -> None:
        super().__init__()
        self.n_series = n_series
        self.n_targets = n_targets
        self.encode_chunk = int(encode_chunk)
        self.backbone_name = str(backbone)
        self.map_hw = MAP_HW
        self.lstm_hidden = int(lstm_hidden)

        self.encoder = _ResNet50Encoder()
        if pretrained:
            self._load_radimagenet(backbone_weights)
        feat_dim = self.encoder.num_features

        self.plane_embed = nn.Parameter(torch.zeros(n_series, feat_dim))
        self.slice_embed = nn.Parameter(torch.zeros(max_depth, feat_dim))
        self.attn_score = nn.Linear(feat_dim, n_targets)
        self.spatial_score = nn.Conv2d(feat_dim, n_targets, kernel_size=1)
        nn.init.zeros_(self.spatial_score.weight)
        nn.init.zeros_(self.spatial_score.bias)

        self.slice_lstm = nn.LSTM(
            input_size=feat_dim,
            hidden_size=self.lstm_hidden,
            num_layers=1,
            batch_first=True,
            bidirectional=True,
        )
        self.lstm_proj = nn.Linear(2 * self.lstm_hidden, feat_dim)
        nn.init.zeros_(self.lstm_proj.weight)
        nn.init.zeros_(self.lstm_proj.bias)

        self.label_embed = nn.Parameter(torch.zeros(n_targets, feat_dim))
        self.norm = nn.LayerNorm(feat_dim)
        self.head = nn.Sequential(
            nn.Linear(feat_dim, head_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.head_out = nn.Linear(head_dim, n_targets)
        with torch.no_grad():
            w0 = self.head_out.weight[0].detach().clone()
            b0 = self.head_out.bias[0].detach().clone()
            self.head_out.weight.copy_(w0.unsqueeze(0).expand_as(self.head_out.weight))
            self.head_out.bias.copy_(b0.expand_as(self.head_out.bias))

        self.stem_mix = nn.Conv2d(3, 3, kernel_size=1, bias=False)
        self._init_stem_mix()
        self.register_buffer(
            "img_mean", torch.zeros(1, 3, 1, 1), persistent=True,
        )
        self.register_buffer(
            "img_std", torch.ones(1, 3, 1, 1), persistent=True,
        )

    def extra_log(self) -> dict[str, float]:
        """Divergence des lignes de head_out vs la 1re (init liee = 0)."""
        w = self.head_out.weight.detach()
        b = self.head_out.bias.detach()
        dw = (w - w[0]).abs().mean()
        db = (b - b[0]).abs().mean()
        return {
            "head_out_w_div": float(dw),
            "head_out_b_div": float(db),
            "label_embed_abs": float(self.label_embed.detach().abs().mean()),
        }

    def _init_stem_mix(self) -> None:
        """out_c = in_G pour les 3 canaux -> R=G=B=k au depart."""
        with torch.no_grad():
            self.stem_mix.weight.zero_()
            self.stem_mix.weight[:, 1, 0, 0] = 1.0

    def _load_radimagenet(self, path: str) -> None:
        p = Path(path)
        if not p.is_file():
            raise FileNotFoundError(
                f"poids RadImageNet introuvables : {p.resolve()}"
            )
        sd = torch.load(p, map_location="cpu", weights_only=False)
        self.encoder.load_state_dict(sd, strict=True)
        print(
            f"RadImageNet ResNet-50 charge : {p} | CAM {MAP_HW}x{MAP_HW} "
            f"| BiLSTM hidden {self.lstm_hidden} | slots {self.n_series} "
            f"| tete par cible (diag, init liee)",
            flush=True,
        )

    def train(self, mode: bool = True):
        super().train(mode)
        for m in self.encoder.modules():
            if isinstance(m, nn.BatchNorm2d):
                m.eval()
        return self

    def encoder_parameters(self):
        return self.encoder.parameters()

    def head_parameters(self):
        for name, p in self.named_parameters():
            if not name.startswith("encoder."):
                yield p

    def _encode_chunks(self, rgb: torch.Tensor) -> torch.Tensor:
        """rgb : (N, 3, H, W) dans [0, 1] -> (N, C, 14, 14)."""
        n = rgb.size(0)
        c = self.encoder.num_features
        if n == 0:
            return rgb.new_zeros(0, c, MAP_HW, MAP_HW)
        outs = []
        chunk = max(1, self.encode_chunk)
        for i in range(0, n, chunk):
            part = rgb[i:i + chunk]
            if self.training:
                feats = checkpoint(self.encoder, part, use_reentrant=False)
            else:
                feats = self.encoder(part)
            outs.append(feats)
        return torch.cat(outs, dim=0)

    def _spatial_pool(self, maps: torch.Tensor, return_attn: bool = False):
        """maps (N, C, H, W) -> gap (N, C), local (N, T, C) [, attn (N, T, H, W)]."""
        n, c, h, w = maps.shape
        gap = maps.mean(dim=(-2, -1))
        logits = self.spatial_score(maps)
        attn = torch.softmax(
            logits.float().reshape(n, self.n_targets, h * w), dim=-1,
        ).to(maps.dtype)
        flat = maps.reshape(n, c, h * w)
        local = torch.einsum("nth,nch->ntc", attn, flat)
        if return_attn:
            return gap, local, attn.reshape(n, self.n_targets, h, w)
        return gap, local

    def _stem_rgb(self, x: torch.Tensor, idx: torch.Tensor) -> torch.Tensor:
        """Triplet A+B pour les tokens valides. x (B,S,D,H,W), idx plat."""
        _b, s, d, _h, _w = x.shape
        bs = idx // (s * d)
        rest = idx % (s * d)
        ss = rest // d
        dd = rest % d
        d_m = (dd - 1).clamp(0, d - 1)
        d_p = (dd + 1).clamp(0, d - 1)
        center = x[bs, ss, dd]
        rgb = torch.stack(
            [
                0.5 + 0.5 * (center - x[bs, ss, d_m]),
                center,
                0.5 + 0.5 * (x[bs, ss, d_p] - center),
            ],
            dim=1,
        )
        rgb = self.stem_mix(rgb)
        return (rgb - self.img_mean) / self.img_std

    def _temporal_delta(self, gaps: torch.Tensor, b: int, s: int,
                        d: int) -> torch.Tensor:
        """gaps (BSD, C) -> residu LSTM (B, S, D, C). Epoch 0 : 0."""
        c = gaps.size(-1)
        seq = gaps.reshape(b * s, d, c)
        with torch.autocast("cuda", enabled=False):
            h, _ = self.slice_lstm(seq.float())
            delta = self.lstm_proj(h)
        return delta.to(dtype=gaps.dtype).reshape(b, s, d, c)

    def _head_logits(self, pooled: torch.Tensor) -> torch.Tensor:
        """pooled (B, T, C) -> logits (B, T). Epoch 0 = tete v20b partagee."""
        pooled = pooled + self.label_embed.to(dtype=pooled.dtype)
        hidden = self.head(self.norm(pooled))
        return self.head_out(hidden).diagonal(dim1=-2, dim2=-1)

    def _slice_pool(self, x: torch.Tensor, valid: torch.Tensor,
                    gaps: torch.Tensor, local: torch.Tensor):
        """gaps (BSD,C), local (BSD,T,C) -> logits (B,T), slice_attn (B,S,D,T)."""
        b, s, d = valid.shape
        delta = self._temporal_delta(gaps, b, s, d)
        pe = self.plane_embed[:s].view(1, s, 1, -1)
        se = self.slice_embed[:d].view(1, 1, d, -1)
        feats = gaps.reshape(b, s, d, -1) + delta + pe + se
        scores = self.attn_score(feats)
        scores = scores.masked_fill(~valid.unsqueeze(-1), -1.0e4)
        scores = scores.reshape(b, s * d, self.n_targets)
        attn = torch.softmax(scores.float(), dim=1).to(feats.dtype)
        tokens = local.reshape(b, s, d, self.n_targets, -1)
        tokens = tokens + delta.unsqueeze(3) + pe.unsqueeze(3) + se.unsqueeze(3)
        tokens = tokens.reshape(b, s * d, self.n_targets, -1)
        pooled = torch.einsum("bnt,bntc->btc", attn, tokens)
        logits = self._head_logits(pooled)
        return logits, attn.reshape(b, s, d, self.n_targets)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x : (B, S, D, H, W) dans [0, 1] -> logits (B, 12)."""
        b, s, d, _h, _w = x.shape
        assert s == self.n_series, f"attendu {self.n_series} series, recu {s}"
        if d > self.slice_embed.size(0):
            raise ValueError(
                f"D={d} > max_depth={self.slice_embed.size(0)} : "
                "augmenter max_depth"
            )

        valid = x.amax(dim=(-2, -1)) > 1e-6
        n_tok = b * s * d
        idx = valid.reshape(-1).nonzero(as_tuple=False).view(-1)
        feat_dim = self.encoder.num_features
        gaps = x.new_zeros(n_tok, feat_dim)
        local = x.new_zeros(n_tok, self.n_targets, feat_dim)
        if idx.numel() > 0:
            maps = self._encode_chunks(self._stem_rgb(x, idx))
            if gaps.dtype != maps.dtype:
                gaps = gaps.to(dtype=maps.dtype)
                local = local.to(dtype=maps.dtype)
            gap, loc = self._spatial_pool(maps)
            gaps[idx] = gap
            local[idx] = loc
        logits, _attn = self._slice_pool(x, valid, gaps, local)
        return logits

    @torch.no_grad()
    def interpret(self, x: torch.Tensor) -> dict[str, torch.Tensor]:
        """Forward eval + cartes. spatial_attn : (B, S, D, T, 14, 14)."""
        self.eval()
        b, s, d, _h, _w = x.shape
        assert s == self.n_series, f"attendu {self.n_series} series, recu {s}"
        valid = x.amax(dim=(-2, -1)) > 1e-6
        n_tok = b * s * d
        idx = valid.reshape(-1).nonzero(as_tuple=False).view(-1)
        feat_dim = self.encoder.num_features
        hw = MAP_HW
        gaps = x.new_zeros(n_tok, feat_dim)
        local = x.new_zeros(n_tok, self.n_targets, feat_dim)
        spatial = x.new_zeros(n_tok, self.n_targets, hw, hw)
        if idx.numel() > 0:
            maps = self._encode_chunks(self._stem_rgb(x, idx))
            if gaps.dtype != maps.dtype:
                gaps = gaps.to(dtype=maps.dtype)
                local = local.to(dtype=maps.dtype)
                spatial = spatial.to(dtype=maps.dtype)
            gap, loc, spat = self._spatial_pool(maps, return_attn=True)
            gaps[idx] = gap
            local[idx] = loc
            spatial[idx] = spat
        logits, slice_attn = self._slice_pool(x, valid, gaps, local)
        delta = self._temporal_delta(gaps, b, s, d)
        pe = self.plane_embed[:s].view(1, s, 1, 1, -1)
        se = self.slice_embed[:d].view(1, 1, d, 1, -1)
        cam_tokens = (
            local.reshape(b, s, d, self.n_targets, -1)
            + delta.unsqueeze(3) + pe + se
        )
        return {
            "logits": logits,
            "slice_attn": slice_attn,
            "spatial_attn": spatial.reshape(
                b, s, d, self.n_targets, hw, hw,
            ),
            "cam_tokens": cam_tokens,
            "valid": valid,
        }


In [ ]:
# --- KneeNet ViT-e, poids ViT-k320 ---
# -*- coding: utf-8 -*-
"""KneeNet-ViT-e — v1 (CAM+BiLSTM) + stem RGB k-1/k/k+1 de d.

Famille CLS (c/d) : le 2.5D aide (contusion 0.66 -> 0.72) mais le gold
reste ~0.74 vs v1 0.80. Le plafond CLS+v8 est sous v1. On recolle le
graphe qui score (CAM patches + LSTM) et on y met le seul levier D
qui a marche.

  - stem : R=k-1, G=k, B=k+1, stem_mix init G (epoch 0 == v1)
  - cartes patch 14x14, CAM + GAP, BiLSTM init 0 (comme v1)
  - LayerScale gele (recette c/d)

Recette : freeze 1, lr_backbone 3e-5 (v1 tenait), re-warmup 1 epoch.
Option `drop_path_rate` (YAML) : stochastic depth timm, OFF pendant
le freeze (epoch 1 = identity e). Defaut 0 = graphe e inchange.
Poids : `weights/dinov3-vitb16-lvd1689m/model.safetensors`
Entree : (B, 6, 24, H, W) grayscale [0, 1]. Slots = v20b.
"""
from __future__ import annotations

from pathlib import Path

import timm
import torch
import torch.nn as nn
from torch.utils.checkpoint import checkpoint

N_TARGETS = 12
DEFAULT_WEIGHTS = "weights/dinov3-vitb16-lvd1689m"
LSTM_HIDDEN = 256
N_PROTOCOL_SLOTS = 6
PATCH_SIZE = 16
N_PREFIX = 5
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)
TIMM_NAME = "vit_base_patch16_dinov3.lvd1689m"


def _weights_file(path: str | Path) -> Path:
    p = Path(path)
    if p.is_dir():
        for name in ("model.safetensors", "pytorch_model.bin"):
            cand = p / name
            if cand.is_file():
                return cand
        raise FileNotFoundError(
            f"aucun checkpoint dans {p.resolve()} "
            "(attendu model.safetensors)"
        )
    if not p.is_file():
        raise FileNotFoundError(f"poids DINOv3 introuvables : {p.resolve()}")
    return p


class _Dinov3VitEncoder(nn.Module):
    """ViT-B/16 DINOv3 -> cartes (N, 768, H/16, W/16)."""

    def __init__(self, weights_path: str = DEFAULT_WEIGHTS,
                 pretrained: bool = True,
                 drop_path_rate: float = 0.0,
                 grad_checkpoint: bool = True) -> None:
        super().__init__()
        kwargs = dict(
            num_classes=0,
            global_pool="",
            dynamic_img_size=True,
        )
        dp = float(drop_path_rate or 0.0)
        if dp > 0:
            kwargs["drop_path_rate"] = dp
        if pretrained:
            ckpt = _weights_file(weights_path)
            if ckpt.suffix == ".pt":
                self.backbone = timm.create_model(
                    TIMM_NAME, pretrained=False, **kwargs,
                )
                sd = torch.load(ckpt, map_location="cpu", weights_only=False)
                self.load_state_dict(sd, strict=True)
                print(
                    f"DINOv3 ViT-B/16 charge : {ckpt} | {len(sd)} tenseurs",
                    flush=True,
                )
            else:
                self.backbone = timm.create_model(
                    TIMM_NAME,
                    pretrained=True,
                    pretrained_cfg_overlay={"file": str(ckpt)},
                    **kwargs,
                )
                print(f"DINOv3 ViT-B/16 charge : {ckpt}", flush=True)
        else:
            self.backbone = timm.create_model(
                TIMM_NAME, pretrained=False, **kwargs,
            )
        self.backbone.set_grad_checkpointing(bool(grad_checkpoint))
        self.num_features = int(self.backbone.num_features)
        self.patch_size = PATCH_SIZE
        self.n_prefix = int(getattr(
            self.backbone, "num_prefix_tokens", N_PREFIX))
        self.drop_path_rate = dp
        self._drop_path: list[tuple[nn.Module, float]] = []
        for m in self.backbone.modules():
            if m.__class__.__name__ == "DropPath":
                self._drop_path.append((m, float(m.drop_prob)))
        if dp > 0:
            print(
                f"drop_path {dp:g} | {len(self._drop_path)} modules "
                f"(off pendant freeze)",
                flush=True,
            )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x (N, 3, H, W) -> (N, C, h, w) patch tokens."""
        tokens = self.backbone.forward_features(x)
        patches = tokens[:, self.n_prefix:]
        n, n_hw, c = patches.shape
        side = int(n_hw ** 0.5)
        if side * side != n_hw:
            raise ValueError(
                f"grille patch non carree : {n_hw} tokens "
                f"(H={x.shape[-2]}, W={x.shape[-1]})"
            )
        return patches.transpose(1, 2).reshape(n, c, side, side)


class KneeNetVitE(nn.Module):
    arch_id = "kneenet_vit_e"

    def __init__(self, backbone: str = TIMM_NAME,
                 n_series: int = N_PROTOCOL_SLOTS,
                 n_targets: int = N_TARGETS, head_dim: int = 512,
                 dropout: float = 0.2, pretrained: bool = True,
                 encode_chunk: int = 4, max_depth: int = 64,
                 backbone_weights: str = DEFAULT_WEIGHTS,
                 lstm_hidden: int = LSTM_HIDDEN,
                 drop_path_rate: float = 0.0,
                 grad_checkpoint: bool = True) -> None:
        super().__init__()
        self.n_series = n_series
        self.n_targets = n_targets
        self.encode_chunk = int(encode_chunk)
        # False sur grosse carte : sinon le ViT est decoupe en petits
        # lancements et le GPU attend Python entre deux blocs.
        self.grad_checkpoint = bool(grad_checkpoint)
        self.backbone_name = str(backbone)
        self.patch_size = PATCH_SIZE
        self.lstm_hidden = int(lstm_hidden)
        self.drop_path_rate = float(drop_path_rate or 0.0)

        self.encoder = _Dinov3VitEncoder(
            weights_path=backbone_weights, pretrained=pretrained,
            drop_path_rate=self.drop_path_rate,
            grad_checkpoint=self.grad_checkpoint,
        )
        self._freeze_layer_scale()
        feat_dim = self.encoder.num_features

        self.stem_mix = nn.Conv2d(3, 3, kernel_size=1, bias=False)
        self._init_stem_mix()

        self.plane_embed = nn.Parameter(torch.zeros(n_series, feat_dim))
        self.slice_embed = nn.Parameter(torch.zeros(max_depth, feat_dim))
        self.attn_score = nn.Linear(feat_dim, n_targets)
        self.spatial_score = nn.Conv2d(feat_dim, n_targets, kernel_size=1)
        nn.init.zeros_(self.spatial_score.weight)
        nn.init.zeros_(self.spatial_score.bias)

        self.slice_lstm = nn.LSTM(
            input_size=feat_dim,
            hidden_size=self.lstm_hidden,
            num_layers=1,
            batch_first=True,
            bidirectional=True,
        )
        self.lstm_proj = nn.Linear(2 * self.lstm_hidden, feat_dim)
        nn.init.zeros_(self.lstm_proj.weight)
        nn.init.zeros_(self.lstm_proj.bias)

        self.norm = nn.LayerNorm(feat_dim)
        self.head = nn.Sequential(
            nn.Linear(feat_dim, head_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(head_dim, 1),
        )
        self.register_buffer(
            "img_mean",
            torch.tensor(IMAGENET_MEAN).view(1, 3, 1, 1),
            persistent=True,
        )
        self.register_buffer(
            "img_std",
            torch.tensor(IMAGENET_STD).view(1, 3, 1, 1),
            persistent=True,
        )
        dp_txt = (f" | drop_path {self.drop_path_rate:g}"
                  if self.drop_path_rate > 0 else "")
        print(
            f"KneeNet-ViT-e | CAM patch {PATCH_SIZE} | BiLSTM "
            f"| RGB k-1/k/k+1{dp_txt} | dim {feat_dim} | slots {self.n_series}",
            flush=True,
        )

    def _init_stem_mix(self) -> None:
        """out = in_G -> epoch 0 identique a v1."""
        with torch.no_grad():
            self.stem_mix.weight.zero_()
            self.stem_mix.weight[:, 1, 0, 0] = 1.0

    @staticmethod
    def _is_layerscale(name: str) -> bool:
        return name.rsplit(".", 1)[-1].startswith("gamma")

    def _freeze_layer_scale(self) -> None:
        n = 0
        for name, p in self.encoder.named_parameters():
            if self._is_layerscale(name):
                p.requires_grad = False
                n += 1
        print(f"LayerScale gele : {n} tenseurs gamma", flush=True)

    def set_encoder_frozen(self, flag: bool) -> None:
        for name, p in self.encoder.named_parameters():
            p.requires_grad = (not flag) and not self._is_layerscale(name)
        self.stem_mix.weight.requires_grad = not flag
        # DropPath stochastique seulement une fois l'encodeur degele,
        # sinon l'epoch 1 n'est plus identity-e.
        for m, p0 in self.encoder._drop_path:
            m.drop_prob = 0.0 if flag else p0

    def encoder_parameters(self):
        for p in self.encoder.parameters():
            if p.requires_grad:
                yield p
        if self.stem_mix.weight.requires_grad:
            yield self.stem_mix.weight

    def head_parameters(self):
        for name, p in self.named_parameters():
            if name.startswith("encoder.") or name.startswith("stem_mix."):
                continue
            yield p

    def _encode_chunks(self, rgb: torch.Tensor) -> torch.Tensor:
        n = rgb.size(0)
        c = self.encoder.num_features
        if n == 0:
            h = rgb.size(-2) // self.patch_size
            w = rgb.size(-1) // self.patch_size
            return rgb.new_zeros(0, c, h, w)
        outs = []
        chunk = max(1, self.encode_chunk)
        for i in range(0, n, chunk):
            part = rgb[i:i + chunk]
            if self.training and self.grad_checkpoint:
                feats = checkpoint(self.encoder, part, use_reentrant=False)
            else:
                feats = self.encoder(part)
            outs.append(feats)
        return torch.cat(outs, dim=0)

    def _spatial_pool(self, maps: torch.Tensor, return_attn: bool = False):
        n, c, h, w = maps.shape
        gap = maps.mean(dim=(-2, -1))
        logits = self.spatial_score(maps)
        attn = torch.softmax(
            logits.float().reshape(n, self.n_targets, h * w), dim=-1,
        ).to(maps.dtype)
        flat = maps.reshape(n, c, h * w)
        local = torch.einsum("nth,nch->ntc", attn, flat)
        if return_attn:
            return gap, local, attn.reshape(n, self.n_targets, h, w)
        return gap, local

    def _stem_rgb(self, x: torch.Tensor, idx: torch.Tensor) -> torch.Tensor:
        """R=k-1, G=k, B=k+1 puis stem_mix. x (B,S,D,H,W), idx plat."""
        _b, s, d, _h, _w = x.shape
        bs = idx // (s * d)
        rest = idx % (s * d)
        ss = rest // d
        dd = rest % d
        d_m = (dd - 1).clamp(0, d - 1)
        d_p = (dd + 1).clamp(0, d - 1)
        rgb = torch.stack(
            [x[bs, ss, d_m], x[bs, ss, dd], x[bs, ss, d_p]],
            dim=1,
        )
        rgb = self.stem_mix(rgb)
        return (rgb - self.img_mean) / self.img_std

    def _temporal_delta(self, gaps: torch.Tensor, b: int, s: int,
                        d: int) -> torch.Tensor:
        c = gaps.size(-1)
        seq = gaps.reshape(b * s, d, c)
        with torch.autocast("cuda", enabled=False):
            h, _ = self.slice_lstm(seq.float())
            delta = self.lstm_proj(h)
        return delta.to(dtype=gaps.dtype).reshape(b, s, d, c)

    def _slice_pool(self, x: torch.Tensor, valid: torch.Tensor,
                    gaps: torch.Tensor, local: torch.Tensor):
        b, s, d = valid.shape
        delta = self._temporal_delta(gaps, b, s, d)
        pe = self.plane_embed[:s].view(1, s, 1, -1)
        se = self.slice_embed[:d].view(1, 1, d, -1)
        feats = gaps.reshape(b, s, d, -1) + delta + pe + se
        scores = self.attn_score(feats)
        scores = scores.masked_fill(~valid.unsqueeze(-1), -1.0e4)
        scores = scores.reshape(b, s * d, self.n_targets)
        attn = torch.softmax(scores.float(), dim=1).to(feats.dtype)
        tokens = local.reshape(b, s, d, self.n_targets, -1)
        tokens = tokens + delta.unsqueeze(3) + pe.unsqueeze(3) + se.unsqueeze(3)
        tokens = tokens.reshape(b, s * d, self.n_targets, -1)
        pooled = torch.einsum("bnt,bntc->btc", attn, tokens)
        logits = self.head(self.norm(pooled)).squeeze(-1)
        return logits, attn.reshape(b, s, d, self.n_targets)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, s, d, _h, _w = x.shape
        assert s == self.n_series, f"attendu {self.n_series} series, recu {s}"
        if d > self.slice_embed.size(0):
            raise ValueError(
                f"D={d} > max_depth={self.slice_embed.size(0)} : "
                "augmenter max_depth"
            )
        valid = x.amax(dim=(-2, -1)) > 1e-6
        n_tok = b * s * d
        idx = valid.reshape(-1).nonzero(as_tuple=False).view(-1)
        feat_dim = self.encoder.num_features
        gaps = x.new_zeros(n_tok, feat_dim)
        local = x.new_zeros(n_tok, self.n_targets, feat_dim)
        if idx.numel() > 0:
            maps = self._encode_chunks(self._stem_rgb(x, idx))
            gap, loc = self._spatial_pool(maps)
            if gaps.dtype != gap.dtype:
                gaps = gaps.to(dtype=gap.dtype)
            if local.dtype != loc.dtype:
                local = local.to(dtype=loc.dtype)
            gaps[idx] = gap
            local[idx] = loc
        logits, _attn = self._slice_pool(x, valid, gaps, local)
        return logits

    @torch.no_grad()
    def interpret(self, x: torch.Tensor) -> dict[str, torch.Tensor]:
        self.eval()
        b, s, d, img_h, img_w = x.shape
        assert s == self.n_series, f"attendu {self.n_series} series, recu {s}"
        valid = x.amax(dim=(-2, -1)) > 1e-6
        n_tok = b * s * d
        idx = valid.reshape(-1).nonzero(as_tuple=False).view(-1)
        feat_dim = self.encoder.num_features
        hw_h, hw_w = img_h // self.patch_size, img_w // self.patch_size
        gaps = x.new_zeros(n_tok, feat_dim)
        local = x.new_zeros(n_tok, self.n_targets, feat_dim)
        spatial = x.new_zeros(n_tok, self.n_targets, hw_h, hw_w)
        if idx.numel() > 0:
            maps = self._encode_chunks(self._stem_rgb(x, idx))
            gap, loc, spat = self._spatial_pool(maps, return_attn=True)
            if gaps.dtype != gap.dtype:
                gaps = gaps.to(dtype=gap.dtype)
            if local.dtype != loc.dtype:
                local = local.to(dtype=loc.dtype)
            if spatial.dtype != spat.dtype:
                spatial = spatial.to(dtype=spat.dtype)
            gaps[idx] = gap
            local[idx] = loc
            spatial[idx] = spat
        logits, slice_attn = self._slice_pool(x, valid, gaps, local)
        delta = self._temporal_delta(gaps, b, s, d)
        pe = self.plane_embed[:s].view(1, s, 1, 1, -1)
        se = self.slice_embed[:d].view(1, 1, d, 1, -1)
        cam_tokens = (
            local.reshape(b, s, d, self.n_targets, -1)
            + delta.unsqueeze(3) + pe + se
        )
        return {
            "logits": logits,
            "slice_attn": slice_attn,
            "spatial_attn": spatial.reshape(
                b, s, d, self.n_targets, hw_h, hw_w,
            ),
            "cam_tokens": cam_tokens,
            "valid": valid,
        }


In [ ]:
# --- KneeNet CoAt-a ---
# -*- coding: utf-8 -*-
"""KneeNet-CoAt-a — baseline CoAtNet 2.5D @224, graphe ViT-e / v20b.

Seul l'encodeur change vs ViT-e : CoAtNet-0 timm (stem conv + 2 stages
MBConv BN + 2 stages Transformer LN), ImageNet-1k, 27M params (~ResNet-50).
Le reste est le graphe qui score deja :

  - stem 2.5D : R=k-1, G=k, B=k+1 (voisins linspace crop130), stem_mix
    1x1 init G -> epoch 0 = coupe grise repliquee (in-distribution)
  - carte finale 7x7 (stride 32 @224) : CAM 12 tetes + GAP
  - BiLSTM profondeur residuel init 0, attention coupe par cible, 6 slots
  - BN encodeur gelee (running stats ImageNet, affine entrainable),
    comme v20b : chunks correles + checkpoint = stats instables

Normalisation = pretrained_cfg timm (0.5 / 0.5 pour les CoAtNet rw),
pas ImageNet. `backbone` YAML = autre CoAtNet timm (poids assortis).
Entree : (B, 6, 24, H, W) grayscale [0, 1]. Slots = v20b.
"""
from __future__ import annotations

from pathlib import Path

import timm
print('timm', timm.__version__, flush=True)
import torch
import torch.nn as nn
from torch.utils.checkpoint import checkpoint

COAT_N_TARGETS = 12
COAT_TIMM_NAME = "coatnet_0_rw_224.sw_in1k"
COAT_DEFAULT_WEIGHTS = "weights/coatnet_0_rw_224-sw_in1k"
COAT_LSTM_HIDDEN = 256
COAT_N_PROTOCOL_SLOTS = 6


def _coat_weights_file(path: str | Path) -> Path:
    p = Path(path)
    if p.is_dir():
        for name in ("model.safetensors", "pytorch_model.bin"):
            cand = p / name
            if cand.is_file():
                return cand
        raise FileNotFoundError(
            f"aucun checkpoint dans {p.resolve()} "
            "(attendu model.safetensors)"
        )
    if not p.is_file():
        raise FileNotFoundError(f"poids CoAtNet introuvables : {p.resolve()}")
    return p


class _CoatEncoder(nn.Module):
    """CoAtNet timm -> cartes (N, C, H/32, W/32) apres LayerNorm finale."""

    def __init__(self, name: str = COAT_TIMM_NAME,
                 weights_path: str = COAT_DEFAULT_WEIGHTS,
                 pretrained: bool = True,
                 img_size: int = 224) -> None:
        super().__init__()
        self.img_size = int(img_size)
        kwargs = dict(num_classes=0, global_pool="", img_size=self.img_size)
        if pretrained:
            ckpt = _coat_weights_file(weights_path)
            self.backbone = timm.create_model(
                name, pretrained=True,
                pretrained_cfg_overlay={"file": str(ckpt)}, **kwargs,
            )
            print(f"CoAtNet charge : {name} | {ckpt} | entree {self.img_size}",
                  flush=True)
        else:
            self.backbone = timm.create_model(name, pretrained=False, **kwargs)
        self.num_features = int(self.backbone.num_features)
        cfg = self.backbone.pretrained_cfg
        self.mean = tuple(float(v) for v in cfg.get("mean", (0.5, 0.5, 0.5)))
        self.std = tuple(float(v) for v in cfg.get("std", (0.5, 0.5, 0.5)))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.backbone.forward_features(x)


class KneeNetCoat(nn.Module):
    arch_id = "kneenet_coat_a"

    def __init__(self, backbone: str = COAT_TIMM_NAME,
                 n_series: int = COAT_N_PROTOCOL_SLOTS,
                 n_targets: int = COAT_N_TARGETS, head_dim: int = 512,
                 dropout: float = 0.2, pretrained: bool = True,
                 encode_chunk: int = 8, max_depth: int = 64,
                 backbone_weights: str = COAT_DEFAULT_WEIGHTS,
                 lstm_hidden: int = COAT_LSTM_HIDDEN,
                 img_size: int = 224) -> None:
        super().__init__()
        self.n_series = n_series
        self.n_targets = n_targets
        self.encode_chunk = int(encode_chunk)
        self.backbone_name = str(backbone)
        self.lstm_hidden = int(lstm_hidden)
        self.img_size = int(img_size)

        self.encoder = _CoatEncoder(
            name=self.backbone_name, weights_path=backbone_weights,
            pretrained=pretrained, img_size=self.img_size,
        )
        feat_dim = self.encoder.num_features

        self.stem_mix = nn.Conv2d(3, 3, kernel_size=1, bias=False)
        self._init_stem_mix()

        self.plane_embed = nn.Parameter(torch.zeros(n_series, feat_dim))
        self.slice_embed = nn.Parameter(torch.zeros(max_depth, feat_dim))
        self.attn_score = nn.Linear(feat_dim, n_targets)
        self.spatial_score = nn.Conv2d(feat_dim, n_targets, kernel_size=1)
        nn.init.zeros_(self.spatial_score.weight)
        nn.init.zeros_(self.spatial_score.bias)

        self.slice_lstm = nn.LSTM(
            input_size=feat_dim,
            hidden_size=self.lstm_hidden,
            num_layers=1,
            batch_first=True,
            bidirectional=True,
        )
        self.lstm_proj = nn.Linear(2 * self.lstm_hidden, feat_dim)
        nn.init.zeros_(self.lstm_proj.weight)
        nn.init.zeros_(self.lstm_proj.bias)

        self.norm = nn.LayerNorm(feat_dim)
        self.head = nn.Sequential(
            nn.Linear(feat_dim, head_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(head_dim, 1),
        )
        self.register_buffer(
            "img_mean", torch.tensor(self.encoder.mean).view(1, 3, 1, 1),
            persistent=True,
        )
        self.register_buffer(
            "img_std", torch.tensor(self.encoder.std).view(1, 3, 1, 1),
            persistent=True,
        )
        print(
            f"KneeNet-CoAt-a | {self.backbone_name} | CAM + BiLSTM "
            f"| RGB k-1/k/k+1 | dim {feat_dim} | slots {self.n_series} "
            f"| norm {self.encoder.mean}/{self.encoder.std}",
            flush=True,
        )

    def _init_stem_mix(self) -> None:
        with torch.no_grad():
            self.stem_mix.weight.zero_()
            self.stem_mix.weight[:, 1, 0, 0] = 1.0

    def train(self, mode: bool = True):
        super().train(mode)
        for m in self.encoder.modules():
            if isinstance(m, nn.BatchNorm2d):
                m.eval()
        return self

    def set_encoder_frozen(self, flag: bool) -> None:
        for p in self.encoder.parameters():
            p.requires_grad = not flag
        self.stem_mix.weight.requires_grad = not flag

    def encoder_parameters(self):
        for p in self.encoder.parameters():
            if p.requires_grad:
                yield p
        if self.stem_mix.weight.requires_grad:
            yield self.stem_mix.weight

    def head_parameters(self):
        for name, p in self.named_parameters():
            if name.startswith("encoder.") or name.startswith("stem_mix."):
                continue
            yield p

    @torch.no_grad()
    def extra_log(self) -> dict[str, float]:
        """Part de |stem_mix| sur chaque coupe d'entree (G seul = pas de 2.5D)."""
        w = self.stem_mix.weight.detach().abs().sum(dim=(0, 2, 3))
        w = w / w.sum().clamp(min=1e-8)
        return {"stem_mix_km1": float(w[0]), "stem_mix_k": float(w[1]),
                "stem_mix_kp1": float(w[2])}

    def _encode_chunks(self, rgb: torch.Tensor) -> torch.Tensor:
        n = rgb.size(0)
        if n == 0:
            h, w = rgb.size(-2) // 32, rgb.size(-1) // 32
            return rgb.new_zeros(0, self.encoder.num_features, h, w)
        outs = []
        chunk = max(1, self.encode_chunk)
        for i in range(0, n, chunk):
            part = rgb[i:i + chunk]
            if self.training:
                feats = checkpoint(self.encoder, part, use_reentrant=False)
            else:
                feats = self.encoder(part)
            outs.append(feats)
        return torch.cat(outs, dim=0)

    def _spatial_pool(self, maps: torch.Tensor, return_attn: bool = False):
        """maps (N, C, H, W) -> gap (N, C), local (N, T, C) [, attn (N, T, H, W)]."""
        n, c, h, w = maps.shape
        gap = maps.mean(dim=(-2, -1))
        logits = self.spatial_score(maps)
        attn = torch.softmax(
            logits.float().reshape(n, self.n_targets, h * w), dim=-1,
        ).to(maps.dtype)
        flat = maps.reshape(n, c, h * w)
        local = torch.einsum("nth,nch->ntc", attn, flat)
        if return_attn:
            return gap, local, attn.reshape(n, self.n_targets, h, w)
        return gap, local

    def _stem_rgb(self, x: torch.Tensor, idx: torch.Tensor) -> torch.Tensor:
        """R=k-1, G=k, B=k+1 puis stem_mix. x (B,S,D,H,W), idx plat."""
        _b, s, d, _h, _w = x.shape
        bs = idx // (s * d)
        rest = idx % (s * d)
        ss = rest // d
        dd = rest % d
        d_m = (dd - 1).clamp(0, d - 1)
        d_p = (dd + 1).clamp(0, d - 1)
        rgb = torch.stack(
            [x[bs, ss, d_m], x[bs, ss, dd], x[bs, ss, d_p]],
            dim=1,
        )
        rgb = self.stem_mix(rgb)
        return (rgb - self.img_mean) / self.img_std

    def _temporal_delta(self, gaps: torch.Tensor, b: int, s: int,
                        d: int) -> torch.Tensor:
        c = gaps.size(-1)
        seq = gaps.reshape(b * s, d, c)
        with torch.autocast("cuda", enabled=False):
            h, _ = self.slice_lstm(seq.float())
            delta = self.lstm_proj(h)
        return delta.to(dtype=gaps.dtype).reshape(b, s, d, c)

    def _slice_pool(self, x: torch.Tensor, valid: torch.Tensor,
                    gaps: torch.Tensor, local: torch.Tensor):
        b, s, d = valid.shape
        delta = self._temporal_delta(gaps, b, s, d)
        pe = self.plane_embed[:s].view(1, s, 1, -1)
        se = self.slice_embed[:d].view(1, 1, d, -1)
        feats = gaps.reshape(b, s, d, -1) + delta + pe + se
        scores = self.attn_score(feats)
        scores = scores.masked_fill(~valid.unsqueeze(-1), -1.0e4)
        scores = scores.reshape(b, s * d, self.n_targets)
        attn = torch.softmax(scores.float(), dim=1).to(feats.dtype)
        tokens = local.reshape(b, s, d, self.n_targets, -1)
        tokens = tokens + delta.unsqueeze(3) + pe.unsqueeze(3) + se.unsqueeze(3)
        tokens = tokens.reshape(b, s * d, self.n_targets, -1)
        pooled = torch.einsum("bnt,bntc->btc", attn, tokens)
        logits = self.head(self.norm(pooled)).squeeze(-1)
        return logits, attn.reshape(b, s, d, self.n_targets)

    def _encode_valid(self, x: torch.Tensor, valid: torch.Tensor,
                      return_attn: bool = False):
        b, s, d, img_h, img_w = x.shape
        n_tok = b * s * d
        idx = valid.reshape(-1).nonzero(as_tuple=False).view(-1)
        feat_dim = self.encoder.num_features
        gaps = x.new_zeros(n_tok, feat_dim)
        local = x.new_zeros(n_tok, self.n_targets, feat_dim)
        spatial = None
        if idx.numel() > 0:
            maps = self._encode_chunks(self._stem_rgb(x, idx))
            pooled = self._spatial_pool(maps, return_attn=return_attn)
            gap, loc = pooled[0], pooled[1]
            gaps = gaps.to(dtype=gap.dtype)
            local = local.to(dtype=loc.dtype)
            gaps[idx] = gap
            local[idx] = loc
            if return_attn:
                spat = pooled[2]
                spatial = spat.new_zeros(n_tok, self.n_targets, *spat.shape[-2:])
                spatial[idx] = spat
        if return_attn and spatial is None:
            spatial = x.new_zeros(n_tok, self.n_targets, img_h // 32, img_w // 32)
        return gaps, local, spatial

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x : (B, S, D, H, W) dans [0, 1] -> logits (B, 12)."""
        _b, s, d, _h, _w = x.shape
        assert s == self.n_series, f"attendu {self.n_series} series, recu {s}"
        if d > self.slice_embed.size(0):
            raise ValueError(
                f"D={d} > max_depth={self.slice_embed.size(0)} : "
                "augmenter max_depth"
            )
        valid = x.amax(dim=(-2, -1)) > 1e-6
        gaps, local, _ = self._encode_valid(x, valid)
        logits, _attn = self._slice_pool(x, valid, gaps, local)
        return logits

    @torch.no_grad()
    def interpret(self, x: torch.Tensor) -> dict[str, torch.Tensor]:
        """Forward eval + cartes. spatial_attn : (B, S, D, T, h, w)."""
        self.eval()
        b, s, d, _h, _w = x.shape
        assert s == self.n_series, f"attendu {self.n_series} series, recu {s}"
        valid = x.amax(dim=(-2, -1)) > 1e-6
        gaps, local, spatial = self._encode_valid(x, valid, return_attn=True)
        logits, slice_attn = self._slice_pool(x, valid, gaps, local)
        delta = self._temporal_delta(gaps, b, s, d)
        pe = self.plane_embed[:s].view(1, s, 1, 1, -1)
        se = self.slice_embed[:d].view(1, 1, d, 1, -1)
        cam_tokens = (
            local.reshape(b, s, d, self.n_targets, -1)
            + delta.unsqueeze(3) + pe + se
        )
        hw_h, hw_w = spatial.shape[-2:]
        return {
            "logits": logits,
            "slice_attn": slice_attn,
            "spatial_attn": spatial.reshape(b, s, d, self.n_targets, hw_h, hw_w),
            "cam_tokens": cam_tokens,
            "valid": valid,
        }


In [ ]:
# --- KneeNet CNX-r7, namespace isole ---
import json as _json
_cnx_src = _json.loads("\"# -*- coding: utf-8 -*-\\n\\\"\\\"\\\"KneeNet-CNX-r7 \\u2014 tete R7 (Heliosli) sur l'entree hybrid025.\\n\\nMeme tenseur que CNX-a : (B, 6, 24, 224, 224) grayscale [0, 1], slots\\nv20b. Le cache px384_v4 n'est pas reconstruit : c'est leur rendu\\nprecalcule (384 px, vues 100 mm, meta DICOM). Ici chaque coupe valide\\nest une fenetre 2.5D (k-1, k, k+1), comme les 3 canaux de R7.\\n\\nGraphe, repris de compact_train.Net (spatial + extent + syn_decomp) :\\n  - ConvNeXt-Small stride 32, drop_path 0.1, carte brute (sans head.norm).\\n  - Attention spatiale par pathologie (requete nulle a l'init = moyenne).\\n  - LayerNorm de tete apres le pooling, par classe.\\n  - Attention par pathologie sur les coupes du study.\\n  - Etendue OA : gamma * log1p(aire/100), gamma nul a l'init.\\n    Aire = produit sagittal x coronal (OA TF) ou axial x sagittal (PF).\\n    Epaisseur de dalle : 3.5 mm x nombre de coupes valides du slot\\n    (le zip crop130 n'a pas le spacing DICOM).\\n  - Synovite = part specifique + beta * stopgrad(logit epanchement).\\n\\nPas de tetes d'etat ni d'auxiliaire fastMRI : les cibles sont v3b (12\\ncolonnes). Ces tetes ne changent pas les logits tant que leur porte\\nvaut 0, et il n'y a pas de labels d'etat pour les apprendre.\\n\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nimport math\\nfrom pathlib import Path\\n\\nimport timm\\nprint('timm', timm.__version__, flush=True)\\nimport torch\\nimport torch.nn as nn\\nimport torch.nn.functional as F\\nfrom torch.utils.checkpoint import checkpoint\\n\\nN_TARGETS = 12\\nTIMM_NAME = \\\"convnext_small.fb_in22k_ft_in1k\\\"\\nDEFAULT_WEIGHTS = \\\"weights/convnext_small-fb_in22k_ft_in1k\\\"\\nN_PROTOCOL_SLOTS = 6\\nHIDDEN = 384\\nOA_MED, OA_LAT, OA_PF = 4, 5, 6\\nSYN, EFF = 8, 7\\n# slots v20b : SAG_FS, COR_FS, AX_FS, SAG_T1, COR_T1, AX_T1\\nSLOT_PLANE = (0, 1, 2, 0, 1, 2)\\nSLOT_FLUID = (1, 1, 1, 0, 0, 0)\\nSLAB_MM = 3.5\\nIMAGENET_MEAN = (0.485, 0.456, 0.406)\\nIMAGENET_STD = (0.229, 0.224, 0.225)\\n\\n\\ndef _weights_file(path: str | Path) -> Path:\\n    p = Path(path)\\n    if p.is_dir():\\n        for name in (\\\"model.safetensors\\\", \\\"pytorch_model.bin\\\"):\\n            cand = p / name\\n            if cand.is_file():\\n                return cand\\n        raise FileNotFoundError(\\n            f\\\"aucun checkpoint dans {p.resolve()} \\\"\\n            \\\"(attendu model.safetensors)\\\"\\n        )\\n    if not p.is_file():\\n        raise FileNotFoundError(f\\\"poids ConvNeXt introuvables : {p.resolve()}\\\")\\n    return p\\n\\n\\nclass _ConvNextEncoder(nn.Module):\\n    \\\"\\\"\\\"Carte stride 32 avant head.norm. drop_path 0.1, comme R7.\\\"\\\"\\\"\\n\\n    def __init__(self, name: str = TIMM_NAME,\\n                 weights_path: str = DEFAULT_WEIGHTS,\\n                 pretrained: bool = True) -> None:\\n        super().__init__()\\n        kwargs = dict(\\n            num_classes=0, global_pool=\\\"\\\", output_stride=32,\\n            drop_path_rate=0.1,\\n        )\\n        if pretrained:\\n            ckpt = _weights_file(weights_path)\\n            self.backbone = timm.create_model(\\n                name, pretrained=True,\\n                pretrained_cfg_overlay={\\\"file\\\": str(ckpt)}, **kwargs,\\n            )\\n            print(f\\\"ConvNeXt charge : {name} | {ckpt} | stride 32 | drop_path 0.1\\\",\\n                  flush=True)\\n        else:\\n            self.backbone = timm.create_model(\\n                name.split(\\\".\\\", 1)[0], pretrained=False, **kwargs,\\n            )\\n        self.num_features = int(self.backbone.num_features)\\n        cfg = self.backbone.pretrained_cfg if pretrained else {}\\n        self.mean = tuple(float(v) for v in cfg.get(\\\"mean\\\", IMAGENET_MEAN))\\n        self.std = tuple(float(v) for v in cfg.get(\\\"std\\\", IMAGENET_STD))\\n\\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\\n        return self.backbone.forward_features(x)\\n\\n\\nclass KneeNet(nn.Module):\\n    arch_id = \\\"kneenet_cnx_r7\\\"\\n\\n    def __init__(self, backbone: str = TIMM_NAME,\\n                 n_series: int = N_PROTOCOL_SLOTS,\\n                 n_targets: int = N_TARGETS, hidden: int = HIDDEN,\\n                 dropout: float = 0.1, pretrained: bool = True,\\n                 encode_chunk: int = 4, max_depth: int = 64,\\n                 backbone_weights: str = DEFAULT_WEIGHTS) -> None:\\n        super().__init__()\\n        self.n_series = n_series\\n        self.n_targets = n_targets\\n        self.encode_chunk = int(encode_chunk)\\n        self.backbone_name = str(backbone)\\n        self.hidden = int(hidden)\\n        self.max_depth = int(max_depth)\\n\\n        self.encoder = _ConvNextEncoder(\\n            name=self.backbone_name, weights_path=backbone_weights,\\n            pretrained=pretrained,\\n        )\\n        d = self.encoder.num_features\\n        norm = self.encoder.backbone.head.norm\\n        if isinstance(norm, nn.Identity):\\n            raise RuntimeError(\\\"ConvNeXt sans head.norm : le pooling R7 en a besoin\\\")\\n        self.sp_head_norm = norm\\n\\n        self.sp_key = nn.Linear(d, 128)\\n        self.sp_pos = nn.Linear(16, 128, bias=False)\\n        nn.init.zeros_(self.sp_pos.weight)\\n        self.sp_q = nn.Parameter(torch.zeros(n_targets, 128))\\n\\n        self.proj = nn.Sequential(nn.Linear(d, self.hidden), nn.LayerNorm(self.hidden))\\n        self.slot = nn.Embedding(10, self.hidden)\\n        self.view = nn.Embedding(6, self.hidden)\\n        self.geom = nn.Linear(4, self.hidden, bias=False)\\n        nn.init.zeros_(self.slot.weight)\\n        nn.init.zeros_(self.view.weight)\\n        nn.init.zeros_(self.geom.weight)\\n        self.drop = nn.Dropout(dropout)\\n        self.win_proj = nn.Linear(self.hidden, self.hidden)\\n        self.q_win = nn.Parameter(torch.randn(n_targets, self.hidden) * 0.02)\\n        self.out_norm = nn.LayerNorm(self.hidden)\\n        self.cls_w = nn.Parameter(torch.randn(n_targets, self.hidden) * 0.02)\\n        self.bias = nn.Parameter(torch.zeros(n_targets))\\n\\n        self.ext_u = nn.Parameter(torch.zeros(n_targets, self.hidden))\\n        self.ext_b = nn.Parameter(torch.full((n_targets,), -4.0))\\n        self.ext_gamma = nn.Parameter(torch.zeros(n_targets))\\n        self.ext_view = nn.Parameter(torch.zeros(n_targets, 6))\\n        em = torch.zeros(n_targets)\\n        em[OA_MED] = em[OA_LAT] = em[OA_PF] = 1.0\\n        self.register_buffer(\\\"ext_mask\\\", em)\\n        self.syn_beta = nn.Parameter(torch.zeros(()))\\n\\n        self.register_buffer(\\n            \\\"img_mean\\\", torch.tensor(self.encoder.mean).view(1, 3, 1, 1),\\n            persistent=True,\\n        )\\n        self.register_buffer(\\n            \\\"img_std\\\", torch.tensor(self.encoder.std).view(1, 3, 1, 1),\\n            persistent=True,\\n        )\\n        self.register_buffer(\\\"slot_plane\\\", torch.tensor(SLOT_PLANE, dtype=torch.long))\\n        self.register_buffer(\\\"slot_fluid\\\", torch.tensor(SLOT_FLUID, dtype=torch.float32))\\n        print(\\n            f\\\"KneeNet-CNX-r7 | {self.backbone_name} | hidden {self.hidden} \\\"\\n            f\\\"| spatial + fenetres + etendue OA + syn_beta \\\"\\n            f\\\"| slots {self.n_series} | dim {d}\\\",\\n            flush=True,\\n        )\\n\\n    def set_encoder_frozen(self, flag: bool) -> None:\\n        for p in self.encoder.parameters():\\n            p.requires_grad = not flag\\n\\n    def encoder_parameters(self):\\n        yield from self.encoder.parameters()\\n\\n    def head_parameters(self):\\n        for name, p in self.named_parameters():\\n            if name.startswith(\\\"encoder.\\\"):\\n                continue\\n            yield p\\n\\n    @torch.no_grad()\\n    def extra_log(self) -> dict[str, float]:\\n        return {\\n            \\\"syn_beta\\\": float(self.syn_beta.detach().cpu()),\\n            \\\"ext_gamma_abs\\\": float(self.ext_gamma.detach().abs().mean()),\\n            \\\"sp_q_abs\\\": float(self.sp_q.detach().abs().mean()),\\n        }\\n\\n    def _stem_rgb(self, x: torch.Tensor, idx: torch.Tensor) -> torch.Tensor:\\n        \\\"\\\"\\\"Trois coupes adjacentes -> 3 canaux. x (B,S,D,H,W).\\\"\\\"\\\"\\n        _b, s, d, _h, _w = x.shape\\n        bs = idx // (s * d)\\n        rest = idx % (s * d)\\n        ss = rest // d\\n        dd = rest % d\\n        d_m = (dd - 1).clamp(0, d - 1)\\n        d_p = (dd + 1).clamp(0, d - 1)\\n        rgb = torch.stack(\\n            [x[bs, ss, d_m], x[bs, ss, dd], x[bs, ss, d_p]],\\n            dim=1,\\n        )\\n        return (rgb - self.img_mean) / self.img_std\\n\\n    def _encode_indexed(self, x: torch.Tensor, idx: torch.Tensor) -> torch.Tensor:\\n        chunk = max(1, self.encode_chunk)\\n        outs = []\\n        for i in range(0, int(idx.numel()), chunk):\\n            rgb = self._stem_rgb(x, idx[i:i + chunk])\\n            if self.training:\\n                feats = checkpoint(self.encoder, rgb, use_reentrant=False)\\n            else:\\n                feats = self.encoder(rgb)\\n            outs.append(feats)\\n        return torch.cat(outs, dim=0)\\n\\n    def _pos_enc(self, h: int, w: int, device: torch.device, dtype: torch.dtype) -> torch.Tensor:\\n        yy, xx = torch.meshgrid(\\n            torch.linspace(-1, 1, h, device=device),\\n            torch.linspace(-1, 1, w, device=device),\\n            indexing=\\\"ij\\\",\\n        )\\n        fr = torch.tensor([1.0, 2.0, 4.0, 8.0], device=device) * (math.pi / 2)\\n        yy = yy.reshape(-1, 1) * fr\\n        xx = xx.reshape(-1, 1) * fr\\n        return torch.cat([yy.sin(), yy.cos(), xx.sin(), xx.cos()], 1).to(dtype)\\n\\n    def spatial_pool(self, maps: torch.Tensor) -> torch.Tensor:\\n        \\\"\\\"\\\"maps (N, d, h, w) -> (N, C, d). Requete nulle = moyenne des positions.\\\"\\\"\\\"\\n        _n, _d, h, w = maps.shape\\n        f = maps.flatten(2).transpose(1, 2)\\n        pe = self._pos_enc(h, w, f.device, f.dtype)\\n        kin = F.layer_norm(f.float(), (f.shape[-1],)).to(f.dtype)\\n        k = torch.tanh(self.sp_key(kin) + self.sp_pos(pe)[None])\\n        att = torch.einsum(\\\"npk,ck->npc\\\", k, self.sp_q.to(k.dtype)).float().softmax(1).to(f.dtype)\\n        g = torch.einsum(\\\"npc,npd->ncd\\\", att, f)\\n        nrm = self.sp_head_norm\\n        g = F.layer_norm(g.float(), (g.shape[-1],), nrm.weight, nrm.bias, nrm.eps).to(f.dtype)\\n        return g\\n\\n    def _window_meta(self, valid: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:\\n        \\\"\\\"\\\"valid (B,S,D) -> mask (B,W), meta (B,W,9).\\\"\\\"\\\"\\n        b, s, d = valid.shape\\n        device = valid.device\\n        rel = torch.linspace(0, 1, d, device=device)\\n        n_valid = valid.flatten(2).sum(-1).clamp(min=1).to(torch.float32)\\n        slot = torch.arange(s, device=device)\\n        plane = self.slot_plane[:s].to(device)\\n        fluid = self.slot_fluid[:s].to(device)\\n        meta = valid.new_zeros(b, s, d, 9, dtype=torch.float32)\\n        meta[..., 0] = slot.view(1, s, 1).float()\\n        meta[..., 2] = rel.view(1, 1, d)\\n        meta[..., 3] = math.log1p(SLAB_MM)\\n        meta[..., 4] = 1.0\\n        meta[..., 5] = fluid.view(1, s, 1)\\n        meta[..., 6] = slot.view(1, s, 1).float()\\n        meta[..., 7] = plane.view(1, s, 1).float()\\n        meta[..., 8] = n_valid.view(b, s, 1)\\n        return valid.reshape(b, s * d), meta.reshape(b, s * d, 9)\\n\\n    def extent_area(self, e: torch.Tensor, meta: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:\\n        \\\"\\\"\\\"e (B,W,C) -> aire proxy (B,C), mm^2. Meme combinaison de plans que R7.\\\"\\\"\\\"\\n        v = 6\\n        sid = meta[..., 6].long()\\n        s_max = max(16, int(sid.max()) + 1)\\n        vid = meta[..., 1].long().clamp(0, v - 1)\\n        q = torch.sigmoid(e.float()) * mask[..., None]\\n        ohv = F.one_hot(sid * v + vid, s_max * v).float() * mask[..., None]\\n        cntv = ohv.sum(1)\\n        mean_v = (\\n            torch.einsum(\\\"bwk,bwc->bkc\\\", ohv, q) / cntv.clamp_min(1)[..., None]\\n        ).view(e.shape[0], s_max, v, -1)\\n        present = (cntv > 0).view(e.shape[0], s_max, v, 1)\\n        wv = self.ext_view.float().t()[None, None].masked_fill(~present, -1e4).softmax(2) * present\\n        mean = (wv * mean_v).sum(2)\\n        oh = F.one_hot(sid, s_max).float() * mask[..., None]\\n        cnt = oh.sum(1)\\n        span = torch.expm1(meta[..., 3].float()).clamp_min(0) * meta[..., 8].float()\\n        span_s = torch.einsum(\\\"bws,bw->bs\\\", oh, span) / cnt.clamp_min(1)\\n        plane_s = (torch.einsum(\\\"bws,bw->bs\\\", oh, meta[..., 7].float()) / cnt.clamp_min(1)).round()\\n        energy = mean * span_s[..., None]\\n        per_plane = []\\n        for pl in (0, 1, 2):\\n            ok = ((plane_s == pl) & (cnt > 0))[..., None]\\n            per_plane.append(energy.masked_fill(~ok, 0.0).amax(1))\\n        sag, cor, ax = per_plane\\n        area = torch.zeros_like(sag)\\n        area[:, OA_MED] = sag[:, OA_MED] * cor[:, OA_MED]\\n        area[:, OA_LAT] = sag[:, OA_LAT] * cor[:, OA_LAT]\\n        area[:, OA_PF] = ax[:, OA_PF] * sag[:, OA_PF]\\n        return area\\n\\n    def _decompose_syn(self, z: torch.Tensor) -> torch.Tensor:\\n        ze = z[:, EFF].detach()\\n        onehot = z.new_zeros(z.shape[1])\\n        onehot[SYN] = 1\\n        return z + (self.syn_beta * ze)[:, None] * onehot[None, :]\\n\\n    def head(self, tok: torch.Tensor, meta: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:\\n        \\\"\\\"\\\"tok (N,C,d) des fenetres valides, dans l'ordre du masque. -> (B,C).\\\"\\\"\\\"\\n        b, w = mask.shape\\n        f = self.proj(tok.float())\\n        emb = (\\n            self.slot(meta[..., 0].long().clamp(0, 9))\\n            + self.view(meta[..., 1].long().clamp(0, 5))\\n            + self.geom(meta[..., 2:6])\\n        )\\n        t = f.new_zeros((b, w) + tuple(f.shape[1:]))\\n        t[mask] = f\\n        t = self.drop(t + emb[:, :, None, :])\\n        s2 = torch.einsum(\\\"bwch,ch->bwc\\\", torch.tanh(self.win_proj(t)), self.q_win).float()\\n        s2 = s2.masked_fill(~mask[:, :, None], float(\\\"-inf\\\"))\\n        a2 = s2.softmax(1).to(t.dtype)\\n        study = self.out_norm(torch.einsum(\\\"bwc,bwch->bch\\\", a2, t))\\n        e_win = (t.float() * self.ext_u[None, None]).sum(-1) + self.ext_b\\n        z = (study * self.cls_w).sum(-1).float() + self.bias\\n        area = self.extent_area(e_win, meta, mask)\\n        z = z + self.ext_gamma * self.ext_mask * torch.log1p(area / 100.0)\\n        return self._decompose_syn(z)\\n\\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\\n        \\\"\\\"\\\"x (B, S, D, H, W) dans [0, 1] -> logits (B, 12).\\\"\\\"\\\"\\n        _b, s, d, _h, _w = x.shape\\n        if s != self.n_series:\\n            raise ValueError(f\\\"attendu {self.n_series} series, recu {s}\\\")\\n        if d > self.max_depth:\\n            raise ValueError(f\\\"D={d} > max_depth={self.max_depth}\\\")\\n        valid = x.amax(dim=(-2, -1)) > 1e-6\\n        mask, meta = self._window_meta(valid)\\n        idx = mask.reshape(-1).nonzero(as_tuple=False).view(-1)\\n        if idx.numel() == 0:\\n            return x.new_zeros(x.shape[0], self.n_targets)\\n        maps = self._encode_indexed(x, idx)\\n        tok = self.spatial_pool(maps)\\n        return self.head(tok, meta, mask)\\n\"")
_cnx_ns = {'__name__': 'kneenet_cnx_r7'}
exec(compile(_cnx_src, 'kneenet_cnx_r7.py', 'exec'), _cnx_ns)
KneeNet = _cnx_ns['KneeNet']
print('KneeNet CNX-r7 isole', KneeNet.arch_id, flush=True)


In [ ]:
import pandas as pd
import time

TARGETS = ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA',
           'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's",
           'Contusion', 'Fracture']


def _cap_chunk(cfg, cap):
    raw = int(cfg.get('encode_chunk', cap))
    chunk = cap if raw > cap else raw
    if chunk != raw:
        print(f'encode_chunk {raw} -> {chunk} (T4)')
    return chunk


def _expect_swa(ckpt, label, epochs):
    if str(ckpt.get('epoch')) != label:
        raise RuntimeError(ckpt.get('epoch'))
    if [int(e) for e in (ckpt.get('swa_epochs') or [])] != list(epochs):
        raise RuntimeError(ckpt.get('swa_epochs'))


def load_vit(weights_path):
    ckpt = torch.load(str(weights_path), map_location='cpu', weights_only=False)
    cfg = ckpt.get('config', {})
    if str(cfg.get('run_name')) != 'vit_k_llrd08_320_v3b_oofmask_v20h_full_oai3':
        raise RuntimeError(cfg.get('run_name'))
    if str(cfg.get('model')) != 'kneenet_vit_k320':
        raise RuntimeError(cfg.get('model'))
    if int(cfg.get('img_size', 0)) != 320:
        raise RuntimeError(cfg.get('img_size'))
    if cfg.get('laterality'):
        raise RuntimeError('vit-k320 ne doit pas etre flipped')
    _expect_swa(ckpt, 'swa-1-2-3', [1, 2, 3])
    model = KneeNetVitE(
        n_series=int(cfg.get('n_series', 6)),
        head_dim=int(cfg.get('head_dim', 512)),
        dropout=float(cfg.get('dropout', 0.4)),
        encode_chunk=_cap_chunk(cfg, 2),
        lstm_hidden=int(cfg.get('lstm_hidden', 256)),
        pretrained=False,
    )
    model.load_state_dict(ckpt['model'], strict=True)
    model.to(DEVICE).eval()
    print('vit', cfg.get('run_name'), 'epoch', ckpt.get('epoch'),
          'gold', round(float(ckpt.get('gold_macro_auc')), 4))
    return model


def load_resnet(weights_path, cls, model_name, run_name, swa_label, swa_epochs):
    ckpt = torch.load(str(weights_path), map_location='cpu', weights_only=False)
    cfg = ckpt.get('config', {})
    if str(cfg.get('run_name')) != run_name:
        raise RuntimeError(cfg.get('run_name'))
    if str(cfg.get('model')) != model_name:
        raise RuntimeError(cfg.get('model'))
    if cfg.get('laterality'):
        raise RuntimeError('ce run ne doit pas etre flipped')
    _expect_swa(ckpt, swa_label, swa_epochs)
    model = cls(
        backbone=cfg.get('backbone', 'resnet50'),
        n_series=int(cfg.get('n_series', 6)),
        head_dim=int(cfg.get('head_dim', 512)),
        dropout=float(cfg.get('dropout', 0.4)),
        encode_chunk=_cap_chunk(cfg, 8),
        lstm_hidden=int(cfg.get('lstm_hidden', 256)),
        pretrained=False,
    )
    model.load_state_dict(ckpt['model'], strict=True)
    model.to(DEVICE).eval()
    print(run_name, 'epoch', ckpt.get('epoch'), 'lat', cfg.get('laterality'),
          'gold', round(float(ckpt.get('gold_macro_auc')), 4))
    return model


model_vit = load_vit(WEIGHTS_VIT)
model_v20h = load_resnet(
    WEIGHTS_V20H, KneeNetV20h, 'kneenet_v20h',
    'v20h_crop145_dino_ep11_v3b_rsna_distil_main_full', 'swa-7-8-9', [7, 8, 9],
)
model_v20b = load_resnet(
    WEIGHTS_V20B, KneeNetV20b, 'kneenet_v20b',
    'v20b_crop130_dino_ep11_oof_v3b_full_oai3', 'swa-1-2-3', [1, 2, 3],
)

def load_coat(weights_path):
    ckpt = torch.load(str(weights_path), map_location='cpu', weights_only=False)
    cfg = ckpt.get('config', {})
    if str(cfg.get('run_name')) != 'coat_a_llrd08_v3b_oofmask_full_oai3':
        raise RuntimeError(cfg.get('run_name'))
    if str(cfg.get('model')) != 'kneenet_coat_a':
        raise RuntimeError(cfg.get('model'))
    if str(cfg.get('backbone')) != 'coatnet_0_rw_224.sw_in1k':
        raise RuntimeError(cfg.get('backbone'))
    if cfg.get('laterality'):
        raise RuntimeError('coat ne doit pas etre flipped')
    if cfg.get('img_size') not in (None, 224):
        raise RuntimeError(cfg.get('img_size'))
    _expect_swa(ckpt, 'swa-1-2-3', [1, 2, 3])
    model = KneeNetCoat(
        backbone=str(cfg.get('backbone')),
        n_series=int(cfg.get('n_series', 6)),
        head_dim=int(cfg.get('head_dim', 512)),
        dropout=float(cfg.get('dropout', 0.4)),
        encode_chunk=_cap_chunk(cfg, 8),
        lstm_hidden=int(cfg.get('lstm_hidden', 256)),
        pretrained=False,
        img_size=224,
    )
    model.load_state_dict(ckpt['model'], strict=True)
    model.to(DEVICE).eval()
    print('coat', cfg.get('run_name'), 'epoch', ckpt.get('epoch'),
          'lat', cfg.get('laterality'),
          'gold', round(float(ckpt.get('gold_macro_auc')), 4))
    return model


model_coat = load_coat(WEIGHTS_COAT)

def load_cnx(weights_path):
    ckpt = torch.load(str(weights_path), map_location='cpu', weights_only=False)
    cfg = ckpt.get('config', {})
    if str(cfg.get('run_name')) != 'cnx_r7_in22k_v3b_oofmask_full':
        raise RuntimeError(cfg.get('run_name'))
    if str(cfg.get('model')) != 'kneenet_cnx_r7':
        raise RuntimeError(cfg.get('model'))
    if str(cfg.get('backbone')) != 'convnext_small.fb_in22k_ft_in1k':
        raise RuntimeError(cfg.get('backbone'))
    if cfg.get('laterality'):
        raise RuntimeError('cnx ne doit pas etre flipped')
    if int(cfg.get('hidden', -1)) != 384:
        raise RuntimeError(cfg.get('hidden'))
    _expect_swa(ckpt, 'swa-7-8-9-10', [7, 8, 9, 10])
    chunk = _cap_chunk(cfg, 4)
    model = KneeNet(
        backbone=str(cfg.get('backbone')),
        n_series=int(cfg.get('n_series', 6)),
        n_targets=12,
        hidden=int(cfg.get('hidden', 384)),
        dropout=float(cfg.get('dropout', 0.1)),
        pretrained=False,
        encode_chunk=chunk,
        max_depth=int(cfg.get('max_depth', 64) or 64),
    )
    model.load_state_dict(ckpt['model'], strict=True)
    model.to(DEVICE).eval()
    print('cnx', cfg.get('run_name'), 'epoch', ckpt.get('epoch'),
          'lat', cfg.get('laterality'),
          'gold', round(float(ckpt.get('gold_macro_auc')), 4))
    return model


model_cnx = load_cnx(WEIGHTS_CNX)

test = pd.read_csv(TEST_CSV)
test_series = pd.read_csv(TEST_SERIES_CSV)
print(len(test), 'etudes | sans TTA | pas de flip | FOV vit/v20b/coat/cnx 130 mm, v20h 145 mm | cnx forward fp32 | 2 GPU')
for t, w in zip(TARGETS, MIX):
    if abs(sum(w) - 1.0) > 1e-6 or min(w) < -1e-9 or len(w) != 5:
        raise RuntimeError((t, w))
    print(f'  {t:<18} vit {w[0]:.2f} v20b {w[1]:.2f} v20h {w[2]:.2f} coat {w[3]:.2f} cnx {w[4]:.2f}')


def _forward(model, x):
    if DEVICE == 'cuda':
        with torch.autocast('cuda', dtype=AMP_DTYPE):
            return model(x)
    return model(x)


import copy
import threading
from concurrent.futures import ThreadPoolExecutor

_TLS = threading.local()


def _center_crop(img, row_mm, col_mm):
    crop_mm = float(getattr(_TLS, 'crop_mm', CROP_MM))
    h, w = img.shape[:2]
    if row_mm is not None and col_mm is not None and row_mm > 0 and col_mm > 0:
        ch = min(h, max(8, int(round(crop_mm / row_mm))))
        cw = min(w, max(8, int(round(crop_mm / col_mm))))
    else:
        ch = min(h, max(8, int(round(h * CROP_REL))))
        cw = min(w, max(8, int(round(w * CROP_REL))))
    y0, x0 = (h - ch) // 2, (w - cw) // 2
    return img[y0:y0 + ch, x0:x0 + cw]


def load_series_dicom(series_dir, img_size=224):
    cache = getattr(_TLS, 'series_cache', None)
    if cache is None:
        cache = {}
        _TLS.series_cache = cache
    key = str(Path(series_dir))
    if key not in cache:
        cache[key] = _series_band(series_dir)
    band = cache[key]
    size = int(img_size)
    out = np.empty((band.shape[0], size, size), dtype=np.uint8)
    for i in range(band.shape[0]):
        sl = band[i]
        if sl.shape != (size, size):
            sl = cv2.resize(sl, (size, size), interpolation=cv2.INTER_AREA)
        out[i] = np.clip(np.asarray(sl, dtype=np.float32) * 255.0, 0, 255).round().astype(np.uint8)
    return out


def _split_gpu(model):
    model.cpu()
    replica = copy.deepcopy(model)
    return model.to('cuda:0').eval(), replica.to('cuda:1').eval()


n_gpu = torch.cuda.device_count() if DEVICE == 'cuda' else 0
if n_gpu < 2:
    raise RuntimeError(('attendu 2 GPU', n_gpu))
model_vit, model_vit_b = _split_gpu(model_vit)
model_v20b, model_v20b_b = _split_gpu(model_v20b)
model_v20h, model_v20h_b = _split_gpu(model_v20h)
model_coat, model_coat_b = _split_gpu(model_coat)
model_cnx, model_cnx_b = _split_gpu(model_cnx)
import gc as _gc
print('gpus mix 2 | cuda:0 cuda:1 | decode 1 etude a la fois', flush=True)
_gc.collect()
packs = {
    'cuda:0': (model_vit, model_v20b, model_v20h, model_coat, model_cnx),
    'cuda:1': (model_vit_b, model_v20b_b, model_v20h_b, model_coat_b, model_cnx_b),
}
_DECODE_LOCK = threading.Lock()


def _trim_ram():
    _gc.collect()
    try:
        import ctypes
        ctypes.CDLL('libc.so.6').malloc_trim(0)
    except Exception:
        pass


def _forward_on(model, x):
    with torch.autocast('cuda', dtype=AMP_DTYPE):
        return model(x)


def _decode_study(uid):
    with _DECODE_LOCK:
        _TLS.series_cache = {}
        srows = test_series[test_series.StudyInstanceUID == uid]
        sdir = TEST_SERIES_DIR / uid
        _TLS.crop_mm = 130.0
        vol_224 = study_to_tensor(sdir, srows, n_series=6, laterality=False, img_size=224)
        vol_vit = study_to_tensor(sdir, srows, n_series=6, laterality=False, img_size=320)
        _TLS.series_cache = {}
        _TLS.crop_mm = 145.0
        vol_h = study_to_tensor(sdir, srows, n_series=6, laterality=False, img_size=224)
        _TLS.series_cache = {}
        _TLS.crop_mm = 130.0
        return vol_224, vol_vit, vol_h


def _score_shard(dev, uids):
    torch.cuda.set_device(dev)
    m_vit, m_v20b, m_v20h, m_coat, m_cnx = packs[dev]
    local = {}
    t0 = time.time()
    for n, uid in enumerate(uids):
        if time.time() - NB_T0 > MIX5_BUDGET_S:
            print(f'MIX5_BUDGET {dev} {n}/{len(uids)}', flush=True)
            break
        vol_224, vol_vit, vol_h = _decode_study(uid)
        if vol_vit.shape[-1] != 320 or vol_224.shape[-1] != 224 or vol_h.shape[-1] != 224:
            raise RuntimeError((uid, vol_vit.shape, vol_224.shape, vol_h.shape))
        x_224 = torch.from_numpy(vol_224).unsqueeze(0).to(dev)
        x_vit = torch.from_numpy(vol_vit).unsqueeze(0).to(dev)
        x_h = torch.from_numpy(vol_h).unsqueeze(0).to(dev)
        del vol_224, vol_vit, vol_h
        with torch.no_grad():
            p_vit = torch.sigmoid(_forward_on(m_vit, x_vit).float())
            p_v20b = torch.sigmoid(_forward_on(m_v20b, x_224).float())
            p_v20h = torch.sigmoid(_forward_on(m_v20h, x_h).float())
            p_coat = torch.sigmoid(_forward_on(m_coat, x_224).float())
            with torch.autocast('cuda', enabled=False):
                p_cnx = torch.sigmoid(m_cnx(x_224.float()).float())
            w = torch.tensor(MIX, dtype=torch.float32, device=dev)
            prob = (
                w[:, 0] * p_vit + w[:, 1] * p_v20b + w[:, 2] * p_v20h
                + w[:, 3] * p_coat + w[:, 4] * p_cnx
            ).float().cpu().numpy()[0]
            del p_vit, p_v20b, p_v20h, p_coat, p_cnx, w
        del x_224, x_vit, x_h
        local[uid] = prob
        if (n + 1) % 20 == 0:
            _trim_ram()
            print(f'{dev} {n + 1}/{len(uids)} | {(time.time() - t0) / 60:.1f} min', flush=True)
    _trim_ram()
    return local


uids = [str(u) for u in test.StudyInstanceUID]
with ThreadPoolExecutor(max_workers=2) as pool:
    fa = pool.submit(_score_shard, 'cuda:0', uids[0::2])
    fb = pool.submit(_score_shard, 'cuda:1', uids[1::2])
    scored = fa.result()
    scored.update(fb.result())
missing = [u for u in uids if u not in scored]
if missing and time.time() - NB_T0 < MIX5_BUDGET_S - 30:
    raise RuntimeError(('etudes manquantes', len(missing), len(uids)))
rows = [[u] + scored[u].tolist() for u in uids if u in scored]
pred = pd.DataFrame(rows, columns=['StudyInstanceUID'] + TARGETS)
print(f'MIX5_COVERAGE {len(pred)}/{len(test)}', flush=True)
pred.head()


In [ ]:
# Scores du mix 5, puis liberation du GPU avant Raptor.
from pathlib import Path as _Out
import gc

out_dir = _Out('/kaggle/working') if _Out('/kaggle/working').exists() else _Out('.')
mix_csv = out_dir / 'mix5_probs.csv'
pred.to_csv(mix_csv, index=False)
print('MIX5_SCORED', len(pred), '|', mix_csv, flush=True)
del model_vit, model_v20b, model_v20h, model_coat, model_cnx
del model_vit_b, model_v20b_b, model_v20h_b, model_coat_b, model_cnx_b
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
try:
    import ctypes
    ctypes.CDLL('libc.so.6').malloc_trim(0)
except Exception:
    pass
print('GPU mix5 libere', flush=True)


In [ ]:
%%writefile /tmp/raptor_run.py
# Authorized scoring run: verify the selected checkpoint before inference.
RUN_INFERENCE = True
if not RUN_INFERENCE:
    raise SystemExit("Static notebook only. Attach the exact checkpoint and competition, enable a T4 GPU, and set RUN_INFERENCE = True to run inference.")

import hashlib
from pathlib import Path
import torch
if not torch.cuda.is_available():
    raise RuntimeError("This scored pipeline requires CUDA; CPU inference is not enabled.")
_weight_name = 'raptor_ft_alldata_t16_blendjev_oai_d96_r384_swa.pt'
_candidates = [
    Path('/kaggle/input/rsna-knee-publication-swa-weights-20261007') / _weight_name,
    Path('/kaggle/input/datasets/nartaa/rsna-knee-publication-swa-weights-20261007') / _weight_name,
    Path('/kaggle/input/rsna-knee-oai-original384-s42-pair') / _weight_name,
    Path('/kaggle/input/datasets/nartaa/rsna-knee-oai-original384-s42-pair') / _weight_name,
]
_weights = next((p for p in _candidates if p.is_file()), None)
if _weights is None:
    raise FileNotFoundError("Attach the exact selected weights; no alternative model is substituted.")
with _weights.open('rb') as _f:
    assert hashlib.file_digest(_f, 'sha256').hexdigest() == '7e5315dad125b99fc65b340b3de41de628e9be51ff5835355dd61c86472244ef', "Checkpoint hash differs"


# %% [code]
#!/usr/bin/env python3
"""Knee MRI: twelve findings from a single model

This notebook takes a knee MRI study and scores twelve findings at once: ACL tear, MCL tear,
medial and lateral meniscus tears, osteoarthritis in the medial, lateral and patellofemoral
compartments, joint effusion, synovitis, a Baker's cyst, bone contusion and fracture. It scores
0.924 on the public leaderboard using one model, with no ensembling and no test-time augmentation.

This is the inference half of the work. The model was trained separately and its weights are
attached as a dataset, so this notebook only loads them and predicts:
https://www.kaggle.com/datasets/dreaddevelopment/raptor-knee-widedense

Where the training labels came from

Worth saying up front, because it shapes everything else. The competition gives you 4,407 studies
but structured labels for only 58 of them. Every other study arrives with a free-text radiology
report and nothing more, so there is very little to train against out of the box.

The labels behind these weights were made by reading those reports with a language model and
turning each into twelve probabilities rather than twelve yes or no answers. A report that says a
tear is suspected becomes a number near 0.8, not a 1, which is a fairer target than forcing every
hedged sentence into a hard label. That yields 4,349 studies to train on. The 58 studies that came
with real labels were never trained on and are used to check the result honestly; the model reaches
0.9167 macro-AUC on them.

Building a fixed input from studies that are all shaped differently

The hard part of this competition is not the network, it is that no two studies look alike. A
study holds several DICOM series shot in different planes, the number of series varies, and the
number of slices in a series varies more. Anything that expects a fixed-size input has to be given
one.

The approach here is to fill five fixed slots per study, always in the same order, for a stack of
64 images:

  18 slices from a sagittal series, preferring a fluid-sensitive one
  14 slices from a second sagittal series, preferring one that is not fluid-sensitive
  12 slices from a coronal series, preferring a fluid-sensitive one
   8 slices from a second coronal series
  12 slices from an axial series

Preferring a fluid-sensitive series for some slots and not for others is deliberate. Fluid-
sensitive sequences show swelling, effusion and acute injury clearly, while the other sequences
show anatomy and cartilage better, and the twelve findings are split across both. If a study has
no series for a slot, the slot is left as zeros and the model is told to skip it rather than being
fed something misleading.

Within a series, slices are taken evenly across 6 to 94 percent of the stack rather than from the
middle. The outer slices are where the collateral ligaments and the lateral meniscus sit, and
cutting them was measurably costing accuracy on exactly those findings.

Every slice is cropped to a 140 mm box around the centre of the image using the pixel spacing from
the DICOM header, then resized to 336 pixels. Cropping by millimetres rather than by pixel count
matters: it means a knee occupies the same fraction of the frame whether the scan was acquired at
0.3 or 0.5 mm per pixel, so the model is not asked to learn scale differences that carry no medical
information.

How the model reads the stack

Three neighbouring slices are stacked into the three channels of one image. The network then sees
a little of what lies above and below the slice in the middle, which is most of the benefit of a 3D
model at the cost of a 2D one. Each of these three-slice windows is passed through a CoAtNet
backbone at 384 pixels.

The windows are combined with an attention layer that has separate weights for each of the twelve
findings. This is the part that matters most. A cruciate tear may be visible on two sagittal slices
while osteoarthritis is spread across many coronal ones, and a single pooled score forces those two
to share one notion of which slices are important. Giving each finding its own attention weights
lets each one draw on the slices that actually show it.

Running it

Scoring uses 42 windows per study. Inference runs in half precision and automatically retries a
study in full precision if it fails, so no study is ever dropped from the submission. The notebook
needs no internet: the backbone is loaded from the attached weights rather than downloaded.
"""
import os, sys, glob, time, json, gc
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
os.environ.setdefault("HF_HUB_DISABLE_TELEMETRY", "1")
import numpy as np
from concurrent.futures import ThreadPoolExecutor
import threading, queue
import torch, torch.nn as nn, torch.nn.functional as F
import timm
# T4 (Turing) cuDNN v9 has fp16/fp32 conv engines but NOT bf16 for these shapes
# ("GET was unable to find an engine..."); benchmark lets it pick a valid algo for
# the fixed (1,24,3,res,res) input.
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True

# ---- fixed config (must match training exactly) -----------------------------
INPUT_RES = 320
INPUT_MODE = "crop"
IMG = 384   # native 384 corpus: the model saw 384 px crops directly, no 336->384 upsample
CROP_MM = 140.0
# 96 slices per study at native 384 px. Must match the corpus the weights were trained on
# (cache/build_corpus96.py).
SLOTS = [("Sagittal", 1, 26), ("Sagittal", 0, 22), ("Coronal", 1, 18),
         ("Coronal", 0, 12), ("Axial", -1, 18)]
MAXS = sum(s[2] for s in SLOTS)                     # 96
K_EVAL = 94   # every window position of the 96-slice test volume (full coverage)
HDR_THREADS = 16   # per prep process: header reads are fuse-latency-bound, not CPU-bound
PX_THREADS = 8     # per prep process: pixel reads of the ~96 picked slices
QUEUE_MAX = 48     # studies in flight between the prep pool and the GPU consumers (RAM bound: ~14 MB each)
NORM = "imagenet"
FASTREAD_VARIANT = 'study_lut'   # kernel v3: research/fastread/readers.py, byte-identical to build_study
FR_HDR, FR_PX = 32, 16
_FASTREAD_B64 = "..."
def _load_fastread():
    import base64 as _b, importlib.util as _iu
    p = '/kaggle/working/_fastread.py' if os.path.isdir('/kaggle/working') else os.path.join(os.path.dirname(os.path.abspath(__file__)), '_fastread.py')
    if not os.path.exists(p):
        open(p, 'wb').write(_b.b64decode(_FASTREAD_B64))
    spec = _iu.spec_from_file_location('_fastread', p); m = _iu.module_from_spec(spec); spec.loader.exec_module(m)
    return m
LAB = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA",
       "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]
_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
_STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

# Three arms: (weights filename, fallback arch, fallback res). ck carries arch+res too.
# Selected 2026-08-19 by greedy forward selection AND exhaustive subset search over a 7-arm
# panel on the 45-study gold set (phase2/blend_panel.py); both agree on this exact set.
# Singles: coatnet384 0.9025 | swinbase384 0.8825 | effv2l480 0.8716.
# Blend {coatnet+swin+effv2l} = 0.9068 (2-arm {coatnet+swin} = 0.9059, coatnet alone 0.9025).
# Dropped as redundant: cnn336 (0.8833, the former champion), cnbase384 (0.8754),
# cnlarge384 (0.8752), maxvit384 (0.8438).
#
# SINGLE ARM: coatnet_rmlp_2_rw_384 retrained on the EXPANDED 4,349-study corpus.
#
# Why one arm and not the 3-arm blend: on the live leaderboard CoAtNet alone scored 0.914 while
# every blend scored 0.914-0.915, so ensembling is worth ~+0.001 there -- the ~+0.010 it showed
# on the old 45-study gold set was gold-set noise. One arm is also 1/3 the kernel runtime.
#
# Corpus expansion: the corpus previously held 3,200 of the 4,349 labelled studies and only 45
# of the 58 gold studies. Rebuilt to 4,407 studies (+37.8% training data, 58-study gate).
#
# Measured on the 58-study gate (the incumbent re-scored on the SAME gate for a fair compare):
#   incumbent CoAtNet (3,155-study corpus) 0.8923
#   this model       (4,349-study corpus) 0.9054   (+0.0131, better in 92.7% of 2000 bootstraps)
# Biggest gains land on the findings that were capping us: Lateral Meniscus +0.071,
# Fracture +0.057, Lateral OA +0.048, Medial Meniscus +0.035, ACL +0.028.
ARMS = [   # our 5-fold CoAtNet (public recipe reproduced on our folds/labels), equal-weight rank-mean
    {"file": "raptor_ft_alldata_t16_blendjev_oai_d96_r384_swa.pt", "arch": "coatnet_rmlp_2_rw_384.sw_in12k_ft_in1k", "res": 384, "w": 1.0},
]


# ============================================================================
# Model -- verbatim from finetune_raptor.py
# ============================================================================
def build_backbone(arch, pretrained=False):
    # maxvit/maxxvit/coatnet are conv-attention hybrids: NO CLS token, NO interpolatable
    # pos-embed -> avg pool. The "vit" substring in "coatnet"/"maxvit" must NOT route them
    # down the ViT path (mirrors finetune_raptor.py exactly).
    hybrid = arch.startswith(("maxvit", "maxxvit", "coatnet", "coat_", "convnext"))
    is_vit = (not hybrid) and any(k in arch for k in ("vit", "deit", "dinov2", "eva", "beit"))
    kw = dict(pretrained=pretrained, num_classes=0, in_chans=3, img_size=INPUT_RES)
    if is_vit:
        kw.update(global_pool="token", dynamic_img_size=True)
    else:
        kw.update(global_pool="avg")
    return timm.create_model(arch, **kw)


class RaptorClassifier(nn.Module):
    def __init__(self, backbone, F_dim=768, n=12, drop=0.2):
        super().__init__()
        self.backbone = backbone
        self.norm = nn.LayerNorm(F_dim)
        self.att = nn.Sequential(nn.Linear(F_dim, 256), nn.Tanh(), nn.Dropout(drop),
                                 nn.Linear(256, n))
        self.clsW = nn.Parameter(torch.zeros(n, F_dim))
        self.clsb = nn.Parameter(torch.zeros(n))
        nn.init.trunc_normal_(self.clsW, std=0.02)
        self.n = n

    def encode(self, x):
        B, K = x.shape[:2]
        # Bound backbone memory; the attention head still sees every window together.
        flat = x.flatten(0, 1)
        f = torch.cat([self.backbone(chunk) for chunk in flat.split(16)], dim=0)
        return f.view(B, K, -1)

    def head(self, feats):
        h = self.norm(feats)
        a = self.att(h)
        a = torch.softmax(a, dim=1)
        pooled = torch.einsum("bkn,bkf->bnf", a, h)
        logits = (pooled * self.clsW).sum(-1) + self.clsb
        return logits

    def forward(self, x):
        return self.head(self.encode(x))


def load_model(pt_path, arch_default, res_default, device, ngpu=1):
    import hashlib
    with open(pt_path, "rb") as stream:
        observed_sha = hashlib.file_digest(stream, "sha256").hexdigest()
    assert observed_sha == '7e5315dad125b99fc65b340b3de41de628e9be51ff5835355dd61c86472244ef', "Bound checkpoint bytes differ"
    print(f"[weights] sha256 {observed_sha}", flush=True)
    ck = torch.load(pt_path, map_location="cpu", weights_only=False)
    arch = ck.get("arch", arch_default)
    assert int(ck.get("res", res_default)) == 384, "Expected original 384 checkpoint"
    ck_res = INPUT_RES
    bb = build_backbone(arch, pretrained=False)
    model = RaptorClassifier(bb, F_dim=bb.num_features)
    model.load_state_dict(ck["model"], strict=True)
    model.eval().to(device)
    # NOTE: DataParallel removed on purpose. On the full hidden test it drove a system-RAM OOM
    # (per-forward module replication over many studies); a single T4 handles K_EVAL=24 windows
    # fine. Arms are also run SEQUENTIALLY (see main) so peak RAM == one model, not two.
    del ck
    gc.collect()
    return model, ck_res


# ============================================================================
# Eval windowing -- verbatim from finetune_raptor.py StudyWindows (train=False)
# ============================================================================
def _eval_centers(mask, D, k):
    valid = np.where(mask > 0)[0]
    if len(valid) < 3:
        valid = np.arange(min(3, D))
    lo, hi = int(valid.min()), int(valid.max())
    cs = [c for c in range(lo + 1, hi) if c - 1 >= lo and c + 1 <= hi]
    if not cs:
        cs = [max(1, min((lo + hi) // 2, D - 2))]
    idx = np.linspace(0, len(cs) - 1, k).round().astype(int)
    return [cs[i] for i in idx]


SLOT_SIZES = [26, 22, 18, 12, 18]
SAGITTAL_SLOTS = (0, 1)

def _slot_index(D):
    b = np.cumsum([0] + SLOT_SIZES)
    if b[-1] != D: raise RuntimeError(f'slot sizes sum to {b[-1]}, volume has {D} slices')
    s_ = np.searchsorted(b, np.arange(D), side='right') - 1
    rel = (np.arange(D) - b[s_]) / np.maximum(1, np.array(SLOT_SIZES)[s_] - 1)
    return s_, rel

def _mirror_tri(tri, cs, D):
    """final-g1-bundle MIRROR_TTA: true L/R mirror of (k,3,H,W) uint8 eval triplets (research/rnd4/mirror/mirror_f0.py). The slot of each
    window's CENTRE slice decides the axis: sagittal (slots 0,1) -> reverse the 3 slice-channels; coronal/axial (2,3,4) -> flip the width.
    Done on uint8 BEFORE the per-channel LUT, i.e. 'mirror the knee, then normalise'."""
    s_, _ = _slot_index(D)
    sag = torch.from_numpy(np.isin(s_[np.asarray(cs, dtype=np.int64)], SAGITTAL_SLOTS)).to(tri.device).view(-1, 1, 1, 1)
    return torch.where(sag, tri.flip(1), tri.flip(-1))

def infer_pair_probs(model, plain, mirrored, device):
    assert plain.shape==mirrored.shape==(94,3,320,320)
    assert all(not m.training for m in model.modules()), "TTA must not update BN or enable dropout"
    p = infer_probs(model, plain, device)
    q = infer_probs(model, mirrored, device)
    return 0.5 * (p + q)

def eval_windows(vol, mask, k, res, norm=NORM, mirror=False):
    D = vol.shape[0]
    cs = _eval_centers(mask, D, k)
    wins = np.empty((len(cs), 3, res, res), np.float32)
    for j, c in enumerate(cs):
        c = max(1, min(c, D - 2))
        tri = np.stack([vol[c - 1], vol[c], vol[c + 1]], 0)
        if mirror:
            tri = _mirror_tri(torch.from_numpy(tri)[None], [c], D)[0].numpy()
        tri = tri.astype(np.float32) / 255.0
        assert tri.shape[-2:] == (IMG, IMG) and res == INPUT_RES
        lo = (IMG - res) // 2
        tri = tri[:, lo:lo + res, lo:lo + res]
        t = torch.from_numpy(np.ascontiguousarray(tri))
        if t.shape[-1] != res:
            t = F.interpolate(t[None], size=(res, res), mode="bilinear",
                              align_corners=False)[0]
        wins[j] = t.numpy()
    x = torch.from_numpy(wins)
    if norm == "imagenet":
        x = (x - _MEAN) / _STD
    return x


_WLUT = {}
def gpu_windows(vol, mask, k, res, dev, norm=NORM, mirror=False):
    """Same windows as eval_windows, built on the GPU from the uint8 volume. Values come from a per-channel
    256-entry table computed on the CPU with eval_windows' own float32 ops, so they are bit-identical."""
    D = vol.shape[0]
    cs = np.array([max(1, min(c, D - 2)) for c in _eval_centers(mask, D, k)])
    tri = torch.from_numpy(np.ascontiguousarray(vol[np.stack([cs - 1, cs, cs + 1], 1)])).to(dev)   # (k,3,IMG,IMG) uint8
    if mirror:
        tri = _mirror_tri(tri, cs, D)
    assert tri.shape[-2:] == (IMG, IMG) and res == INPUT_RES
    lo = (IMG - res) // 2
    tri = tri[:, :, lo:lo + res, lo:lo + res]
    key = (str(dev), norm)
    if key not in _WLUT:
        v = torch.from_numpy((np.arange(256, dtype=np.uint8).astype(np.float32) / 255.0))
        t = torch.stack([v, v, v], 0).view(3, 256, 1)
        if norm == 'imagenet':
            t = (t - _MEAN.view(3, 1, 1)) / _STD.view(3, 1, 1)
        _WLUT[key] = t.view(3, 256).contiguous().to(dev)
    L = _WLUT[key]
    x = torch.stack([L[c][tri[:, c].long()] for c in range(3)], 1)                   # exact float32 values
    if x.shape[-1] != res:
        if norm == 'imagenet':   # resize must happen BEFORE normalisation, as in eval_windows
            L0 = _WLUT.setdefault((str(dev), 'none'), torch.from_numpy(np.arange(256, dtype=np.uint8).astype(np.float32) / 255.0).to(dev))
            x = L0[tri.long()]
            x = F.interpolate(x, size=(res, res), mode='bilinear', align_corners=False)
            x = (x - _MEAN.to(dev)) / _STD.to(dev)
        else:
            x = F.interpolate(x, size=(res, res), mode='bilinear', align_corners=False)
    return x


@torch.no_grad()
def infer_probs(model, xwins, device):
    assert str(device).startswith("cuda"), "Qualified inference requires CUDA/fp16"
    x = xwins.unsqueeze(0).to(device)
    with torch.autocast("cuda", dtype=torch.float16):
        logits = model(x).float()
        assert torch.isfinite(logits).all(), "Nonfinite fp16 logits"
        o = torch.sigmoid(logits)
    return o[0].cpu().numpy()


def rankpct(x):                                   # per-column percentile rank in [0,1]
    order = x.argsort(0).argsort(0).astype(np.float64)
    return order / max(1, (x.shape[0] - 1))


# ============================================================================
# Preprocessing -- verbatim from kprep2/dino_preprocess.py, retargeted to TEST
# ============================================================================
def _make_reader():
    import pydicom, cv2
    from pydicom.pixel_data_handlers.util import apply_modality_lut

    def _hdr(f):
        try:
            h = pydicom.dcmread(f, stop_before_pixels=True)
            iop = getattr(h, 'ImageOrientationPatient', None)
            ipp = getattr(h, 'ImagePositionPatient', None)
            if iop is not None and ipp is not None and len(iop) == 6:
                r = np.array(iop[:3], float); c = np.array(iop[3:], float)
                n = np.cross(r, c); pos = float(np.dot(np.array(ipp, float), n))
            else:
                pos = float(getattr(h, 'InstanceNumber', 0) or 0)
            ps = getattr(h, 'PixelSpacing', None); ps = float(ps[0]) if ps is not None else 0.5
            return (pos, f, ps, True)
        except Exception:
            return (0.0, f, 0.5, False)

    def order_and_meta(sdir):
        # Header reads are latency-bound on Kaggle's mounted competition data (~240k opens on the hidden
        # set); a thread pool hides that latency. Records and sort are unchanged -> identical ordering.
        fs = glob.glob(sdir + "/*.dcm")
        with ThreadPoolExecutor(max_workers=HDR_THREADS) as ex:
            out = list(ex.map(_hdr, fs))
        recs = [(pos, f, ps) for pos, f, ps, _ in out]; ps_list = [ps for _, _, ps, ok in out if ok]
        recs.sort(key=lambda x: x[0])
        med_ps = float(np.median(ps_list)) if ps_list else 0.5
        return [(f, ps) for _, f, ps in recs], med_ps

    def read_px(f):
        d = pydicom.dcmread(f)
        a = apply_modality_lut(d.pixel_array, d).astype(np.float32)
        if str(getattr(d, 'PhotometricInterpretation', '')) == 'MONOCHROME1':
            a = a.max() - a
        return a

    def mm_crop_resize(a, ps):
        h, w = a.shape; cpx = int(round(CROP_MM / max(ps, 1e-3)))
        cpx = min(cpx, min(h, w)); y0 = (h - cpx) // 2; x0 = (w - cpx) // 2
        a = a[y0:y0 + cpx, x0:x0 + cpx]
        return cv2.resize(a, (IMG, IMG), interpolation=cv2.INTER_AREA)

    return order_and_meta, read_px, mm_crop_resize


def _pick_series_for_slot(rows, plane, fluid, used):
    cands = [r for r in rows if r['Anatomical_Plane'] == plane and r['SeriesInstanceUID'] not in used]
    if fluid in (0, 1):
        pref = [r for r in cands if int(r.get('Fluid_Sensitive', 0) or 0) == fluid]
        if pref:
            return pref[0]
    return cands[0] if cands else None


def build_study(sid, ser_records, tsdir, reader):
    order_and_meta, read_px, mm_crop_resize = reader
    rows = ser_records.get(sid, [])
    vol = np.zeros((MAXS, IMG, IMG), np.uint8); idx = 0; used = set()
    for plane, fluid, k in SLOTS:
        r = _pick_series_for_slot(rows, plane, fluid, used)
        if r is None:
            idx += k; continue
        used.add(r['SeriesInstanceUID'])
        files, med_ps = order_and_meta(f"{tsdir}/{sid}/{r['SeriesInstanceUID']}")
        if not files:
            idx += k; continue
        # wide span: the collateral ligaments and lateral meniscus live in the
        # peripheral slices the old 0.15-0.85 crop threw away. Must match the corpus
        # the weights were trained on (build_corpus80.py, SPAN).
        n = len(files); lo, hi = int(n * 0.02), int(n * 0.98) - 1; hi = max(hi, lo)
        picks = np.linspace(lo, hi, k).round().astype(int) if n > 1 else [0] * k
        sel = [files[min(p, n - 1)] for p in picks]
        def _px(fp_ps):
            fp, ps = fp_ps
            try: return read_px(fp), ps
            except Exception: return None, med_ps
        with ThreadPoolExecutor(max_workers=PX_THREADS) as ex:
            got = list(ex.map(_px, sel))                    # order preserved
        arrs = [a for a, _ in got]; pss = [ps for _, ps in got]
        valid = [a for a in arrs if a is not None]
        if valid:
            allpx = np.concatenate([a.ravel() for a in valid])
            loq, hiq = np.percentile(allpx, [2.0, 98.0])
        else:
            loq, hiq = 0.0, 1.0
        for a, ps in zip(arrs, pss):
            if idx >= MAXS: break
            if a is None: idx += 1; continue
            aw = np.clip((a - loq) / (hiq - loq + 1e-6), 0, 1)
            aw = mm_crop_resize(aw, ps if ps > 0 else med_ps)
            vol[idx] = (aw * 255).astype(np.uint8); idx += 1
        if idx >= MAXS: break
    mask = (vol.reshape(MAXS, -1).sum(1) > 0).astype(np.uint8)
    return vol, mask



# ---- parallel preprocessing helpers (module level so they pickle under fork) ----
_SER = None; _TSDIR = None; _READER = None
def _pre_init():
    global _READER
    _READER = _load_fastread().make_builder(FASTREAD_VARIANT, IMG, SLOTS, hdr_threads=FR_HDR, px_threads=FR_PX)
def _pre_one(sid):
    try:
        vol, mask = _READER(sid, _SER, _TSDIR)
    except Exception as e:
        print(f"  [pre] {sid[:16]} FAILED ({type(e).__name__}: {e})", flush=True)
        vol = np.zeros((MAXS, IMG, IMG), np.uint8); mask = np.zeros(MAXS, np.uint8)
    return sid, vol, mask

# ============================================================================
# Test-root discovery + weights + main
# ============================================================================
def find_test_root():
    cands = ["/kaggle/input/competitions/rsna-knee-abnormality-detection",
             "/kaggle/input/rsna-knee-abnormality-detection"]
    for b in cands:
        if os.path.exists(b + "/test.csv"):
            return b
    for d, _, f in os.walk("/kaggle/input"):
        if "test.csv" in f and (os.path.isdir(d + "/test_series") or os.path.isdir(d + "/test_images")):
            return d
    for d, _, f in os.walk("/kaggle/input"):
        if "test.csv" in f:
            return d
    raise RuntimeError("no test root under /kaggle/input")


def find_weight_file(fname):
    # direct dataset mounts first; NEVER recursive-glob the competitions DICOM tree.
    direct = [f"/kaggle/input/raptor-knee-arms/{fname}",
              f"/kaggle/input/raptor-knee-arms/1/{fname}",
              f"/kaggle/input/raptor-cnn336/{fname}"]
    for p in direct:
        if os.path.exists(p):
            return p
    for d in sorted(glob.glob("/kaggle/input/*/")):
        # skip the competition data wherever Kaggle mounts it (/kaggle/input/competitions/<slug> OR /kaggle/input/<slug>):
        # recursively globbing its DICOM tree hung a commit run for 20+ min (2026-09-26)
        if "competition" in d.lower() or "abnormality-detection" in d or any(
                os.path.exists(os.path.join(d, x)) for x in ("test_series", "train_images", "train.csv", "test.csv")):
            continue
        hits = glob.glob(os.path.join(d, "**", fname), recursive=True)
        if hits:
            return hits[0]
    raise RuntimeError(f"{fname} not found under /kaggle/input")


def main():
    import pandas as pd
    t0 = time.time()
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    ngpu = torch.cuda.device_count()
    print(f"device {dev} | gpus {ngpu} | torch {torch.__version__}", flush=True)

    ROOT = find_test_root()
    tsdir = ROOT + "/test_series"
    if not os.path.isdir(tsdir):
        tsdir = ROOT + "/test_images"
    print("test root:", ROOT, "| series dir:", tsdir, flush=True)

    test = pd.read_csv(ROOT + "/test.csv"); test["StudyInstanceUID"] = test["StudyInstanceUID"].astype(str)
    test_ids = test["StudyInstanceUID"].tolist()
    tser = pd.read_csv(ROOT + "/test_series.csv")
    tser["StudyInstanceUID"] = tser["StudyInstanceUID"].astype(str)
    tser["SeriesInstanceUID"] = tser["SeriesInstanceUID"].astype(str)
    SER = {k: v.to_dict("records") for k, v in tser.groupby("StudyInstanceUID")}
    print(f"test studies {len(test_ids)} | test series {len(tser)}", flush=True)

    sub_cols = ["StudyInstanceUID"] + LAB
    ssub = os.path.join(ROOT, "sample_submission.csv")
    if os.path.exists(ssub):
        sub_cols = list(pd.read_csv(ssub, nrows=1).columns)

    N = len(test_ids); A = len(ARMS)
    arm_probs = [np.full((N, len(LAB)), 0.5, np.float32) for _ in range(A)]
    pos = {sid: i for i, sid in enumerate(test_ids)}

    # ---- 2. GPU consumers: one thread per GPU, every arm loaded on every GPU (per-study batch of 1,
    #         so the math is identical to the sequential kernel; only the wall-clock changes) ----
    xla = None
    try:
        import torch_xla; import torch_xla.core.xla_model as xm
        dev = torch_xla.device(); xla = xm
        print("device: XLA/TPU", dev, flush=True)
    except Exception as e:
        print("XLA unavailable:", type(e).__name__, str(e)[:80], "| device", dev, flush=True)
    devs = [f"cuda:{g}" for g in range(ngpu)] if (xla is None and ngpu > 0) else [dev]
    models = {}
    for d in devs:
        models[d] = []
        for a, arm in enumerate(ARMS):
            model, res = load_model(find_weight_file(arm["file"]), arm["arch"], arm["res"], d)
            models[d].append((model, res))
        print(f"[gpu {d}] loaded {A} arm(s) | {time.time()-t0:.0f}s", flush=True)
    q = queue.Queue(maxsize=QUEUE_MAX); done_n = [0]; lock = threading.Lock()

    def consumer(d):
        while True:
            item = q.get()
            if item is None: q.task_done(); return
            sid, vol, mask = item; i = pos[sid]
            try:
                for a, (model, res) in enumerate(models[d]):
                    xw = eval_windows(vol, mask, k=K_EVAL, res=res, norm=NORM) if xla is not None else gpu_windows(vol, mask, K_EVAL, res, d)
                    assert xla is None, "Anatomical L/R TTA requires CUDA/fp16"
                    xm = gpu_windows(vol, mask, K_EVAL, res, d, norm=NORM, mirror=True)
                    arm_probs[a][i] = infer_pair_probs(model, xw, xm, d)
                    del xm
            except Exception as e:
                print(f"  [gpu {d}] study {i} {sid[:16]} FALLBACK ({type(e).__name__}: {e})", flush=True)
            with lock:
                done_n[0] += 1
                if done_n[0] % 200 == 0 or done_n[0] == N:
                    print(f"  [gpu] {done_n[0]}/{N} | {time.time()-t0:.0f}s", flush=True)
            q.task_done()

    threads = [threading.Thread(target=consumer, args=(d,), daemon=True) for d in devs]
    for th in threads: th.start()

    # ---- 3. prep every study ONCE in a process pool and stream it to the GPUs as it lands ----
    import multiprocessing as mp
    global _SER, _TSDIR, _READER
    _SER, _TSDIR = SER, tsdir
    nproc = max(4, min(96, (os.cpu_count() or 8) - 8))
    t_pre = time.time()
    with mp.get_context("fork").Pool(nproc, initializer=_pre_init) as pool:
        for i, (sid, vol, mask) in enumerate(pool.imap_unordered(_pre_one, test_ids, chunksize=2)):
            q.put((sid, vol, mask))
            if (i + 1) % 200 == 0 or i + 1 == N:
                print(f"  [pre] {i+1}/{N} | {time.time()-t0:.0f}s", flush=True)
    print(f"[pre] done: {N} studies in {time.time()-t_pre:.0f}s (procs {nproc} x fastread {FASTREAD_VARIANT} {FR_HDR}/{FR_PX} threads)", flush=True)
    for _ in devs: q.put(None)
    q.join()
    for th in threads: th.join()
    print(f"[gpu] done: {done_n[0]}/{N} studies on {len(devs)} device(s) | {time.time()-t0:.0f}s", flush=True)
    _raw = pd.DataFrame(arm_probs[0].astype(np.float32), columns=LAB)   # [blend] raw probabilities; a failed study stays 0.5
    _raw.insert(0, "StudyInstanceUID", test_ids)
    _raw.to_csv("/kaggle/working/raptor_raw.csv", index=False)
    for d in devs:
        for m, _ in models[d]: del m
    models.clear(); gc.collect()

    # WEIGHTED rank-mean blend across the test set, per finding (the offline recipe).
    # Weights come from ARMS[*]["w"] and are normalised here, so dropping/adding an arm can
    # never silently change the scale. Falls back to equal weights if none are declared.
    _w = np.array([float(a.get("w", 1.0)) for a in ARMS], dtype=np.float64)
    _w = _w / _w.sum()
    print(f"[blend] weighted rank-mean w={dict(zip([a['file'] for a in ARMS], _w.round(4)))}", flush=True)
    ranks = np.tensordot(_w, np.stack([rankpct(np.clip(p, 0, 1)) for p in arm_probs]),
                         axes=(0, 0))                                          # (N,12) in [0,1]
    if not np.isfinite(ranks).all():
        ranks[~np.isfinite(ranks)] = 0.5

    sub = pd.DataFrame(ranks.astype(np.float32), columns=LAB)
    sub.insert(0, "StudyInstanceUID", test_ids)
    sub = sub[sub_cols]
    assert list(sub.columns) == sub_cols, "column order drift"
    assert sub["StudyInstanceUID"].tolist() == test_ids, "row identity drift"
    assert np.isfinite(sub[LAB].values).all()
    out = "/kaggle/working/raptor_sub.csv"   # [blend] Raptor stage output; the blend cell writes submission.csv
    sub.to_csv(out, index=False)
    print("wrote", out, "|", len(sub), "rows x", len(sub.columns), "cols", flush=True)
    print(sub.head().to_string(index=False), flush=True)
    print(f"DONE {time.time()-t0:.0f}s", flush=True)


if __name__ == "__main__":
    main()


In [ ]:
# Stage 1: public Raptor model (checkpoint cell + scored source above, run as its own process so its GPU memory is
# released before stage 2).  A failed Raptor stage stops the notebook: no silent single-model submission.
import os, sys, time, subprocess
NB_T0 = globals().get('NB_T0', time.time())
import gc as _gc
import ctypes as _ct
_gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
try:
    _ct.CDLL('libc.so.6').malloc_trim(0)
except Exception:
    pass
_t384 = time.time()
rc = subprocess.run([sys.executable, '-u', '/tmp/raptor_run.py'], timeout=2 * 3600).returncode
if rc != 0:
    raise RuntimeError('Raptor stage failed with exit code %d' % rc)
for f in ('raptor_sub.csv', 'raptor_raw.csv'):
    assert os.path.isfile('/kaggle/working/' + f), f + ' missing'
RAPTOR_SECONDS = time.time() - _t384
print('RAPTOR_STAGE_DONE seconds %.0f' % RAPTOR_SECONDS, flush=True)
print('RAPTOR384_SEUL', flush=True)
subprocess.run(['nvidia-smi', '--query-gpu=index,memory.used', '--format=csv,noheader'], check=False)


In [ ]:
# Rang 80/20 : 80 % mix5, 20 % Raptor-384.
import numpy as np
import pandas as pd
from pathlib import Path

W_MIX5, W_RAPTOR = 0.80, 0.20
work = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('.')
mix = pd.read_csv(work / 'mix5_probs.csv', dtype={'StudyInstanceUID': str})
raw = pd.read_csv(work / 'raptor_raw.csv', dtype={'StudyInstanceUID': str})
labels = [c for c in mix.columns if c != 'StudyInstanceUID']
assert set(labels) <= set(raw.columns), 'colonnes raptor incompletes'
mix = mix.drop_duplicates('StudyInstanceUID').set_index('StudyInstanceUID')
raw = raw.drop_duplicates('StudyInstanceUID').set_index('StudyInstanceUID')
order = [str(u) for u in test.StudyInstanceUID]
mix_a = mix.reindex(order)
raw_a = raw.reindex(order)[labels]
present = raw_a.notna().all(axis=1)
rap_fail = present & np.isclose(raw_a.to_numpy(dtype=float), 0.5).all(axis=1)
rap_fail = rap_fail | ~present
print(
    'BLEND80 mix5', int(mix_a[labels[0]].notna().sum()),
    'raptor384', int((~rap_fail).sum()), 'de', len(order),
    'poids mix5 0.80 raptor384 0.20',
    flush=True,
)


def _rank_ok(series, ok):
    out = pd.Series(np.nan, index=series.index, dtype=np.float64)
    if bool(ok.any()):
        out.loc[ok] = series.loc[ok].rank(method='average', pct=True)
    return out


cols = {}
for c in labels:
    mr = _rank_ok(mix_a[c], mix_a[c].notna())
    rr = _rank_ok(raw_a[c], ~rap_fail & raw_a[c].notna())
    blended = W_MIX5 * mr + W_RAPTOR * rr
    blended = blended.fillna(mr).fillna(rr).fillna(0.5)
    cols[c] = blended.astype(np.float32)
sub = pd.DataFrame(cols)
sub.insert(0, 'StudyInstanceUID', order)
dest = work / 'submission.csv'
sub.to_csv(dest, index=False)
assert len(sub) == len(order) and np.isfinite(sub[labels].to_numpy()).all()
print('BLEND80_OK', dest, 'lignes', len(sub), flush=True)
sub.head()
